In [ ]:
import yt
import numpy as np
import matplotlib.pyplot as plt
import os
from IPython.display import HTML
from scipy.optimize import curve_fit
from skimage.transform import resize
from imageio.v3 import imread

from python_src.amrex_io import read_amrex_data, read_csv
from python_src.swift_orlandini import swift_theoretical_C0, swift_theoretical_sigma, swift_theoretical_xi, swift_et_al_1996_thermodynamic_model, fit_swift_phi0
from python_src.mixed_system import spherically_averaged_structure_factor, radial_equilibration, make_bins
from python_src.interface_analysis import interface_height, bin_height_spectrum_1D, calculate_surface_tension_profile
from python_src.droplet_analysis import pressure_jump, droplet_radius_mass, droplet_radius_profile_1d, axial_radii

figures = "figures/"
figheight = 4
os.makedirs(figures, exist_ok = True)
DPI = 300
# colors = ['#d53e4f','#fc8d59','#fee08b','#e6f598','#99d594','#3288bd'] # https://colorbrewer2.org/#type=diverging&scheme=Spectral&n=6
# colors = ['#e41a1c','#377eb8','k','#984ea3','#ff7f00','#4daf4a'] # https://colorbrewer2.org/#type=qualitative&scheme=Set1&n=6
# colors = ['#7fc97f','#beaed4','#fdc086','k','#386cb0','#f0027f']
# colors = ['r', '']
colors = [
    "#1f77b4",  # Blue - Good for primary data categories, stands out.
    "#ff7f0e",  # Orange - High contrast with blue, useful for secondary categories.
    "#2ca02c",  # Green - Pleasant and natural, works well with other colors.
    "#d62728",  # Red - Strong and eye-catching, useful for emphasis.
    "#9467bd",  # Purple - Distinct from others, good for additional categories.
    "#8c564b",  # Brown - Neutral and earthy, provides variety.
    "#e377c2",  # Pink - Soft but noticeable, helps with accessibility.
    "#7f7f7f",  # Gray - Neutral, useful for de-emphasized data.
    "#bcbd22",  # Yellow-Green - High visibility, works well with dark backgrounds.
    "#17becf"   # Cyan - Bright and distinguishable, complements the palette.
]
linestyles = ['-', '--', ':', '-.', (0, (5, 5)), (0, (5, 2, 1, 2))]
markers = ['o', 's', '^', 'D', 'x', '*']
linewidth = 1.5
markersize = 10

# LB things
cs2_ideal = 1/3

plt.ioff()

In [ ]:
# def interface_location_spherical(density, level, step_size = 1):
#     verts, _, _, _ = measure.marching_cubes(density, level = level, step_size = step_size) # interface vertex pointcloud
#     center = verts.mean(axis = 0) # center of mass 
#     shifted = verts - center[np.newaxis, :] # shifting rectlinear coordinate center to com

#     # converting from rectilinear to spherical coordinates
#     x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]
#     r = np.sqrt(x**2 + y**2 + z**2)
#     theta = np.arccos(z / r)
#     phi = np.arctan2(y, x)
#     phi[phi < 0] += 2 * np.pi # correcting the bounds of phi
#     shifted_spherical = np.array([r, theta, phi]).T

#     return shifted_spherical, verts

# def coefficents_spherical_harmonics(coordinates, lmax):
#     # calculating the spherical harmonic expansion
#     cols = np.sum(2*np.arange(0, lmax+1, 1)+1)
#     rows = coordinates.shape[0]
#     A = np.empty((rows, cols), dtype = np.complex128)

#     curr_col = 0
#     # d_zeta_lm = {}
#     for l in range(lmax+1):
#         for m in range(-l, l+1, 1):
#             # d_zeta_lm[f"{l, m}"] = 0
#             for n_curr_vertex in range(cols):
#                 theta = coordinates[n_curr_vertex, 1]
#                 phi = coordinates[n_curr_vertex, 2]
#                 A[n_curr_vertex, curr_col] = sph_harm_y(l, m, theta, phi)
#             curr_col += 1

#     f = coordinates[:, 0] - coordinates[:, 0].mean()
#     f = f.astype(np.complex128)
#     c, residuals, rank, s = np.linalg.lstsq(A, f, rcond=None)
    
#     # curr_col = 0
#     # for l in range(lmax+1):
#     #     for m in range(-l, l+1, 1):
#     #         d_zeta_lm[f"{l, m}"] = c[curr_col]
#     #         curr_col += 1
        
#     return c, A

# def spherical_harmonic_expansion(density, level = 0, step_size = 1, lmax = 8, debug = False):
#     # extraction of the interface location in spherical coordinates using the marching cubes algorithm
#     shifted_spherical, interface_vertices = interface_location_spherical(density, level, step_size)
#     zeta_coefficients, spherical_harmonic_vals = coefficents_spherical_harmonics(shifted_spherical, lmax)
#     if debug:
#         return [zeta_coefficients, shifted_spherical, spherical_harmonic_vals]
#     else:
#         return zeta_coefficients

# Thermodynamic model validation

We will utilize the theory derived in [Cahn and Hilliard 1958](https://doi.org/10.1063/1.1744102) to identify the expressions for the thermodynamic coexistence densities $\phi_0$, surface tension $\sigma$ and interface width $\zeta$. The surface tension is defined as the difference between the free energy and the bulk free energy, 

$$\sigma = \int_{-\infty}^{\infty} f_0(\mathbf{r}) + \frac{\kappa}{2}(\nabla \rho)^2 + \frac{\kappa}{2}(\nabla \phi)^2 - f_0^{crit} d\mathbf{r}$$

We will also define the chemical potential of the system for $\rho$ and $\phi$, as $\mu_\rho = \frac{\partial f_0}{\partial \rho}$ and $\mu_\phi = \frac{\partial f_0}{\partial \phi}$. Some assumptions I will be making are that, $\rho = 1$ at all points, meaning that $\nabla\rho = 0$. We will also assume that the temperature of the system is around the critical point, $T \sim T_c$. 

The Gibbs Duhem relation for our system is defined as $f_0 = \rho\mu_\rho + \phi\mu_\phi$. From our first assumption, we can simplify this to $f_0 = \phi\mu_\phi$. At the critical point, $f_0^{crit} = \phi\mu_\phi(1, \phi_0)$. We can also express $f_0(\mathbf{r})$ in this fashion, redefining it as $f_0(\mathbf{r}) = \phi\mu_\phi(1, \phi(\mathbf{r}))$. Rewriting Equation 1 to calculate the difference in value from the bulk and local free energies after substitution of the Gibbs Duhem relation we have

$$\sigma = \int_{-\infty}^{\infty} \Delta f_0(\mathbf{r}) + \frac{\kappa}{2}(\nabla \phi)^2 d\mathbf{r}$$

where $\Delta f_0(\mathbf{r}) = \phi(\mu_\phi(1, \phi(\mathbf{r})) - \mu_\phi(1, \phi_0))$

## Critical expansion

Using Euler-Lagrange, $\Delta f_0(\mathbf{r}) = \kappa(\nabla \phi)^2$ as $x \rightarrow \infty$. We substitute this solution into the above expression to get, 

$$\sigma = 2\int_{-\infty}^{\infty} \Delta f_0(\mathbf{r}) d\mathbf{r}$$

If we substitude the Euler lagrange relation into the surface tension, we obtain an expression for surface tension as a function of $\phi$

$$\sigma = \sqrt{2} \int_{-\phi_0}^{\phi_0} \sqrt{\kappa \Delta f_0} d\phi$$

Writing down our free energy definitions again, we have 

$$ f_0 = \frac{\chi}{4}(1 - \phi^2) - T + \frac{T}{2}[(1 + \phi)\ln{\frac{1 + \phi}{2}} + (1 - \phi)\ln{\frac{1 - \phi}{2}}]$$

$$ \mu_{\phi} = -\frac{\chi}{2}(\phi) +\frac{T}{2}\ln{\frac{1 + \phi}{1 - \phi}}$$

We Taylor expand $f_0$ around the critical order parameter and temperature, $\phi_c$ and $T_c$ and obtain the following expression, identical to that from [Cahn and Hilliard 1958](https://doi.org/10.1063/1.1744102). $\phi_c = 0$ in the model that we are using.

$$\Delta f_0 = \Delta f(\phi, T) - \Delta f(\phi_0, T) = -\beta(T_c - T)(\phi^2 - \phi_0^2) + \gamma(\phi^4 - \phi_0^4)$$

The coefficients of the expansion, $\beta$ and $\gamma$, are defined as

$$\beta = \frac{\partial^3 f_0}{\partial T \partial \phi^2 2!} = \frac{1}{2(1 - \phi_c^2)} = 0.5$$

$$\gamma = \frac{\partial^4 f_0}{\partial \phi^4 4!} = \frac{2T_c(3\phi_c^2 + 1)}{(1 - \phi_c)^2} = \frac{T_c}{12}$$

These subsitutions then lead to the solution for $\phi_0$

$$\phi_0 = \pm \sqrt{\frac{\beta (T_c - T)}{2\gamma}} = \sqrt{\frac{3(T_c - T)}{T_c}}$$

$$\Delta f_0 = \frac{T_c}{12}(\phi_0^2 - \phi)^2$$

Substituting the expressions above into the integral for surface tension, we can calculate a theoretical expression for the surface tension, 

$$\sigma = \frac{2\sqrt{\kappa}}{3\gamma}(\beta (T_c - T))^{1.5} = \frac{8\sqrt{\kappa}}{T_c}(\frac{(T_c - T)}{2})^{1.5}$$

To calculate the interface width, we begin with solving the euler lagrange relation

$$\frac{\partial \phi}{\partial x} = \sqrt{\frac{2 \Delta f_0}{\kappa}}$$

Once we integrate the differential equation above, we obtain the solution to the profile of the interface, 

$$\phi = \phi_0 \tanh{\sqrt{\frac{2\gamma}{\kappa}}\phi_0 x} = \phi_0 \tanh{\sqrt{\frac{T_c - T}{2 \kappa}}x}$$

This results in predicted properties of the coexistence order parameter $\phi_0$, surface tension $\sigma$ and the interface width $\zeta$

$$\phi_0 = \pm \sqrt{\frac{3(T_c - T)}{T_c}}$$

$$\sigma = \frac{8\sqrt{\kappa}}{T_c}\left(\frac{T_c - T}{2}\right)^{1.5}$$

$$\zeta = \sqrt{\frac{\kappa}{T_c - T}}$$

## Full expression


The transcendental expression,

$$\frac{\chi}{T}(2C_1 - 1) = \log{\frac{C_1}{1 - C_1}}$$

can be solved to calculate the coexistence densities of each component. The thermodyanmic model is defined using $n = \rho = C_1 + C_2$ and $\Delta n = \phi = C_1 - C_2$. If we assume that $n = 1$, we can rewrite $\phi = 2C_1 - 1$. We can use the Newton Raphson Method to solve for the value of $C_1$ and hence $\phi$, which for the coexistence densities we term $\phi_0$. We then compare the calculated values of $\phi_0$ to the values predicted by the critical point expansion above and a power law fit to the data.

In [ ]:
chi_s = np.linspace(2.001, 5.01, 1001)
ce_s = np.zeros_like(chi_s)

for ix in np.ndindex(ce_s.shape):
    chi = chi_s[ix]
    guess = 0.99
    ce = swift_theoretical_C0(chi)
    ce_s[ix] = ce

phi0_s = 2*ce_s - 1

fig, ax = plt.subplots(1, 1, figsize = (4, 4))

idx = 0
ax.plot(chi_s, phi0_s, label = "Theory", color = colors[idx], marker = "None", ls = linestyles[idx])

idx += 1
phi0_fit = lambda x, a, b, c: b*(x-c)**a
popt1, pcov1 = curve_fit(phi0_fit, chi_s, phi0_s, p0 = (0.5, 0.1, 1.99))
phi0_fit_theory = phi0_fit(chi_s, *popt1)
ax.plot(chi_s, phi0_fit_theory, label = "Fit", color = colors[idx], marker = "None", ls = linestyles[idx])

idx += 1
crit_phi0 = lambda x: np.sqrt(3 - 6/x)
crit_phi0_theory = crit_phi0(chi_s)
ax.plot(chi_s, crit_phi0_theory, label = "Critical point\n expansion", color = colors[idx], marker = "None", ls = linestyles[idx])

idx_chi_crit_max = np.where(crit_phi0_theory >= phi0_fit_theory)[0][1:]
chi_crit_max = chi_s[idx_chi_crit_max][0]
ax.axvline(x = chi_crit_max, ymin = 0, ymax = 1, color = "k", ls = "-", lw = 0.5)
# print(chi_crit_max)

idx_chi_fit_max = np.where(phi0_fit_theory >= 0.97)[0]
popt2 = np.polyfit(chi_s[idx_chi_fit_max], phi0_s[idx_chi_fit_max], 1)
chi_fit_max = chi_s[idx_chi_fit_max][0]
ax.axvline(x = chi_fit_max, ymin = 0, ymax = 1, color = "k", ls = "-", lw = 0.5)
# print(chi_fit_max)

ax.set_xlabel(r"$\chi$")
ax.set_ylabel(r"$\phi_c$")
ax.legend()
ax.set_title("Comparison of values of " + r"$\phi_c$" + "\ncalculated using different techniques")
# plt.close()
fig.savefig(f"{figures}/model-coexistence_compare.png", dpi = DPI)

To allow initialization of demixed system easily in the code, we compare the coexistence densities at various $\chi/T$ values obtained from the critical expansion and the transcendental theoretical expression and a fit obtained using a function of form $\phi_c = B(\chi/T - C)^A$ where $A, B, C$ are fit parameters. The critical expansion is close to theory at $\chi/T < 2.25$. However above that the critical expansion diverges from the value of the theoretical expression. The fit expression for $\phi_c$ using a single function fits the theoretical expression well. To improve it further a piece wise expansion is used, with boundaries identified as when the critical expansion diverges and when the original fit is larger than the theoretical value.

\begin{equation}
\phi_c = 
\begin{cases} 
      \sqrt{3 - \frac{6}{\frac{\chi}{T}}} &  2 \leq \frac{\chi}{T} \leq 2.27 \\
      - 0.133(\frac{\chi}{T})^2 + 1.030\frac{\chi}{T} - 1.039 & 2.27 < \frac{\chi}{T} \leq 4.15\\
      0.0248\frac{\chi}{T} + 0.863 & \frac{\chi}{T} > 4.15
\end{cases}
\end{equation}

Comparing this piecewise function to the theory formula, we see good agreement at all $\chi/T$ values.

In [ ]:
ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

idx = 0
ax.plot(chi_s, phi0_s, label = "Theory", color = colors[idx], marker = "None", ls = linestyles[idx])

phi_fit_func = np.zeros_like(chi_s)
for i in np.ndindex(chi_s.size):
    phi_fit_func[i] = fit_swift_phi0(chi_s[i])
idx += 1
ax.plot(chi_s, phi_fit_func, label = "Fit", color = colors[idx], marker = "None", ls = linestyles[idx])

ax.set_xlabel(r"$\chi$")
ax.set_ylabel(r"$\phi_c$")
ax.legend()
ax.set_title("Comparison of values of " + r"$\phi_c$" + "\nfrom theory and fitting")

fig.savefig(f"{figures}/model-coexistence_fit.png", dpi = DPI)

This expression will be used in the code when initializing the system so that the initial state of the system if beginning from a phase separated system will not have to undergo a long diffusion phase and will result in greater numerical stability. Expressions for the surface tension can also be calculated from the free energy functional. We first define a surface tension difference from the bulk phases, $\sigma_r$

\begin{equation}

\sigma_b = \int^{c_1}_{c_2} \sqrt{\frac{2c}{\frac{\chi}{T}}\ln{\frac{c}{c_1}} + \frac{2(1-c)}{\frac{\chi}{T}}\ln{\frac{1 - c}{1 - c_1}} - 2(c - c_1)^2 } dc
\end{equation}

This expression is derived from the integral of the free energy difference between components integrated over the area. We multiply $\sigma_b$ by the interfacial contribution, $\sqrt{\frac{\chi \kappa}{T}}$ to obtain the following expression for the theoretical surface tension of the free energy functional. 

$$ \sigma_T =  \sqrt{\frac{\chi \kappa}{T}} \sigma_b $$

This expression must be numerically integrated. Thus, we would like to calculate a mathematical fit of $\sigma_T$ to $\frac{\chi}{T}$ at various $\kappa$ values. This expression is derived by substituting equation 3.12 in the 1958 Cahn Hilliard paper and substituting $\kappa$ for $\kappa/2$. Equation 3.12 is then substituted into Equation 3.19 with the addition of Eqaution 3.16. 

To verify the model implementation, the surface tension obtained at various $\chi/T$ values is plotted below. We restrict out parameter space to values under $\chi/T < 2.5$ as larger values caused simulation instability. We obtain the surface tension by fitting the Young-Laplace equation to the pressure difference between the interior and exterior of droplets of various radii. We compare the surface tension obtained from the simulation to that obtained from the full expression and the critical point expansion below.

### Comparing the effect of changing T

In [ ]:
# chi_real = np.array(["2.05", "2.1", "2.15", "2.2", "2.25", "2.3", "2.35", "2.4", "2.45", "2.5"])
chi_real = np.array(["2.05", "2.1", "2.15", "2.2", "2.25", "2.3", "2.35", "2.45"])
kappa = 0.01
T = 0.2
R_reals = np.array(["0.30", "0.35", "0.40"])

savedir = f"./validation/young_laplace/T_{T}"

L = 32
boxDim = np.array([L, L, L])

RS, CHIS = np.meshgrid(R_reals, chi_real)

radiis = np.zeros_like(RS, dtype = object)
times = np.zeros_like(RS, dtype = object)
# pressures = np.zeros_like(RS, dtype = float)

for iy, ix in np.ndindex(RS.shape):
    R = RS[iy, ix]
    chi = CHIS[iy, ix]

    curr_path = f"{savedir}/chi_{chi}/kappa_{kappa}/R_{R}"
    if not os.path.exists(curr_path):
        continue
    ts = yt.load(f"{curr_path}/hydro_plt*")
    curr_radii = np.zeros(len(ts))
    curr_times = np.zeros(len(ts))
    for i, ds in enumerate(ts):
        profile = read_amrex_data(ds, boxDim)

        C1 = (profile["rho"] + profile["phi"])/2 
        curr_radii[i] = droplet_radius_mass(C1)
        curr_times[i] = int(ds.current_time)
    
    radiis[iy, ix] = curr_radii
    times[iy, ix] = curr_times     

In [ ]:
rows = 2
cols = len(chi_real)//rows + len(chi_real)%rows

fig, axs = plt.subplots(rows, cols, figsize = (cols*figheight//2*cols/rows, figheight//2*rows*cols/rows))
axs = axs.flatten()

for iy, ix in np.ndindex(RS.shape):
    ax = axs[iy]

    curr_time = times[iy, ix]
    curr_radii = radiis[iy, ix]

    ax.plot(curr_time, curr_radii, label = "R = " + R_reals[ix], color = colors[ix], marker = markers[ix], markerfacecolor = "None")

    ax.set_title(r"$\chi$ = " + chi_real[iy])
    ax.legend()
    ax.set_xlabel("Timestep")
    ax.set_ylabel("Radius")

fig.suptitle(f"$T = {T}$")

fig.tight_layout()
plt.close()
# fig

In [ ]:
# ar = 1.25
# fig, axs = plt.subplots(1, 3, figsize = (3*figheight*ar, figheight))

# chi_real = np.array(["2.05", "2.1", "2.15", "2.2", "2.25", "2.3", "2.35", "2.45"])
# kappa = 0.01
# T = ["0.2", "0.3", "0.4", "0.5"]
# R_reals = np.array(["0.30", "0.35", "0.40"])

# L = 32
# boxDim = np.array([L, L, L])

# savedir = f"./validation/young_laplace/"

# for i in range(len(R_reals)):
#     ax = axs[i]
#     for j in range(len(T)):
#         ls = []
#         for k in range(len(chi_real)):
#             curr_path = f"{savedir}/T_{T[j]}/chi_{chi_real[k]}/kappa_{kappa}/R_{R_reals[i]}"
#             if not os.path.exists(curr_path):
#                 continue
#             ds = yt.load(f"{curr_path}/hydro_plt*")[-1]
#             profile = read_amrex_data(ds, boxDim)
#             C1 = (profile["rho"] + profile['phi'])/2
#             # r = droplet_radius_mass(C1)
#             fit_params, _, _ = droplet_radius_profile_1d(C1)
#             ls.append(fit_params[0])
#         slc = slice(0, len(ls))
        
#         ax.plot(chi_real.astype(float)[slc], ls, label = f"T = {T[j]}", 
#                 color = colors[j], marker = markers[j], linestyle = linestyles[j],
#                 markerfacecolor = "None", ms = markersize)

#     ax.axhline(y = float(R_reals[i])*L, color = "lime", label = r"$R_d^{start}$")
#     ax.set_xlabel(r"$\chi/T$")
#     ax.set_ylabel(r"$R_d^{end}$")
#     ax.set_ylim(top = float(R_reals[i])*L*1.025)

#     # ax.set_title(f"$R_d^{{start}} = {float(R_reals[i])*L}$")
#     ax.legend(ncol = 3)

# # fig.suptitle(f"$T = {T}$")
# fig.suptitle("Comparison of steady state droplet radius with T and " + r"$\chi/T$")
# fig.tight_layout()
# plt.close()
# # fig

In [ ]:
chi_real = np.array(["2.05", "2.1", "2.15", "2.2", "2.25", "2.3", "2.35", "2.45"])
kappa = 0.01
T = 0.2
R_reals = np.array(["0.30", "0.35", "0.40"])

savedir = f"./validation/young_laplace/T_{T}"

L = 32
boxDim = np.array([L, L, L])

RS, CHIS = np.meshgrid(R_reals, chi_real)

radiis = np.zeros_like(RS, dtype = float)
pressures = np.zeros_like(RS, dtype = float)
sanity_check = np.zeros_like(RS, dtype = float)

ar = 1.25
cols = 1
fig, ax = plt.subplots(1, cols, figsize = (cols*ar*figheight, figheight), layout = "constrained")

chi_theory = chi_real.astype(float)
sigma_theory = np.zeros_like(chi_theory)

for i, chi in enumerate(chi_theory):
    sigma_theory[i] = swift_theoretical_sigma(chi, kappa)

idx = 0
ax.plot(chi_theory, sigma_theory, label = "Theory", color = colors[idx], ls = linestyles[idx], lw = linewidth)

# sigma_crit = np.array([swift_critical_sigma(1.0, chi*T, T, kappa)[0] for chi in chi_theory.astype(float)])

# idx += 1
# ax.plot(chi_theory, sigma_crit, label = "Landau\n expansion", 
#         color = colors[idx], ls = linestyles[idx], 
#         marker = "None", markerfacecolor = "None", ms = markersize
# )

for iy, ix in np.ndindex(RS.shape):
    R = RS[iy, ix]
    chi = CHIS[iy, ix]

    curr_path = f"{savedir}/chi_{chi}/kappa_{kappa}/R_{R}"
    if not os.path.exists(curr_path):
        continue
    ts = yt.load(f"{curr_path}/hydro_plt*")
    ds = ts[-1]
    profile = read_amrex_data(ds, boxDim)
    
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho + phi)/2
    radiis[iy, ix] = droplet_radius_mass(C1)

    pb = 1/3*(rho + profile['mf4'])
    
    # # derived from moments
    Pxx = pb + 1/3*profile['mf5']
    Pyy = pb - 1/6*profile['mf5'] + 1/2*profile['mf6']
    Pzz = pb - 1/6*profile['mf5'] - 1/2*profile['mf6']

    scp = (Pxx + Pyy + Pzz)/3
    
    pressures[iy, ix] = pressure_jump(scp)

    sanity_check_pressure = rho*T
    sanity_check_pressure += rho*cs2_ideal
    sanity_check[iy, ix] = pressure_jump(sanity_check_pressure)

sigma_real = np.zeros_like(chi_real, dtype = float)
sigma_sanity = np.zeros_like(chi_real, dtype = float)

for i in range(radiis.shape[0]):
    x = 2/radiis[i].copy()
    y = pressures[i].copy()

    popt = np.polyfit(x, y, deg = 1)
    sigma_real[i] = popt[0]

    y = sanity_check[i].copy()
    popt = np.polyfit(x, y, deg = 1)
    sigma_sanity[i] = popt[0]

idx += 1
ax.plot(chi_real.astype(float), sigma_real, label = "Simulation", 
        color = colors[idx], ls = linestyles[idx], 
        marker = markers[idx], markerfacecolor = "None", ms = markersize
)

# idx += 1
# ax.plot(chi_real.astype(float), sigma_sanity, label = "Sanity check", 
#         color = colors[idx], ls = linestyles[idx], 
#         marker = markers[idx], markerfacecolor = "None", ms = markersize
# )

ax.set_xlabel(r"$\frac{\chi}{T}$")
ax.set_ylabel(r"$\sigma$")
ax.legend()
# ax.set_yscale("log")
ax.set_title("Comparing theory and simulated "+r"$\sigma$"+"\nat " + f"$\kappa$={kappa}" + f", T = {T}")

plt.close()
fig.savefig(f"{figures}/model-surface_tension.png", dpi = DPI)

In [ ]:
fig

At low $\chi/T$ values, we see good match between the predicted and real values and good agreement between the critical expansion and theory at all values. The simulated values show good agreement with theory at low $\chi/T$ values, but at larger $\chi/T$ values large differences between theory and simulations emerge.

A second implementation test would be to calculate the interface width($\zeta_T$) of the thermodynamic model and compare that to what is obtained from the simulation. From theory, the full expression for the theoretical interface width is given by,

$$ \zeta_T = 2\sqrt{\frac{\kappa}{\frac{\chi}{T}}} \left[ -1 - \frac{2 \ln{4C_1(1 - C_1)} }{\frac{\chi}{T}(1 - 2C_1)^2 }   \right]^{-0.5}  $$

To assess the interface width from simulations, we simulate a flat interface and fit the expected interface width $\zeta$ using a hyperbolic tangent, 

$$\phi(x) = \phi_c\tanh{\frac{x - h_{int}}{\sqrt{2} \zeta}}$$

From fitting the hyperbolic tangent, $\zeta$ is obtained and plotted against $\chi/T$ below.

In [ ]:
chi_real = np.array(["2.05", "2.1", "2.15", "2.2", "2.25", "2.3", "2.35", "2.4", "2.45", "2.5"])
kappa = 0.01
T = 0.2

savedir = f"./validation/interface_width/T_{T}"

nx = 128
ny = 1
nz = 16
boxDim = np.array([nx, ny, nz])
slc = np.s_[nx//4:3*nx//4, ny//2, nz//2]

ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (figheight*ar, figheight), layout = "constrained")

chi_theory = np.linspace(float(chi_real[0]), float(chi_real[-1]), 101)
interface_width_theory = np.zeros_like(chi_theory)
for i, chi in enumerate(chi_theory):
    interface_width_theory[i] = swift_theoretical_xi(swift_theoretical_C0(chi), chi, kappa)

idx = 0
ax.plot(chi_theory, interface_width_theory, label = "Theory", 
        color = colors[idx], ls = linestyles[idx], 
        marker = "None", lw = linewidth)

# interface_width_crit = np.sqrt(kappa/(chi_theory/2*T - T))
# interface_width_crit = np.array([swift_critical_sigma(1.0, chi*T, T, kappa)[1]/4 for chi in chi_theory.astype(float)])

# idx += 1
# ax.plot(chi_theory, interface_width_crit, label = "Landau expansion", 
#         color = colors[idx], ls = linestyles[idx], 
#         marker = "None", lw = linewidth)

interface_widths = np.zeros((len(chi_real), 2))

profiles = np.zeros((chi_real.size, 3*nx//4- nx//4))
surface_tensions = np.zeros((chi_real.size))

for i in range(len(chi_real)):
    chi = float(chi_real[i])*T

    curr_path = f"{savedir}/chi_{chi_real[i]}/kappa_{kappa}/"
    if not os.path.exists(curr_path):
        continue
    ts = yt.load(f"{curr_path}/hydro_plt*")
    ds = ts[-1]

    profile = read_amrex_data(ds, boxDim)
    phi_sim = profile["phi"][slc]
    profiles[i] = phi_sim

    midpoint = (nx - 1)/2
    x_idxs = np.arange(nx//4, 3*nx//4) - midpoint

    C0 = swift_theoretical_C0(chi/T)
    xi = swift_theoretical_xi(C0, chi/T, kappa)
    surface_tensions[i] = calculate_surface_tension_profile(x_idxs, phi_sim, kappa)

    popt, pcov = curve_fit(lambda x, phi0, xi: -phi0*np.tanh((x)/(np.sqrt(2)*xi)), x_idxs, phi_sim, p0 = (2*C0-1, 1))
    interface_widths[i, 0] = popt[-1]

    interface_widths[i, 1] = np.sqrt(kappa/(chi/2 - T))

idx += 1
ax.plot(chi_real.astype(float), interface_widths[:, 0], label = "Simulation", 
        color = colors[idx], ls = linestyles[idx], marker = markers[idx], 
        markerfacecolor = "None", ms = markersize)

ax.set_xlabel(r"$\frac{\chi}{T}$")
ax.set_ylabel(r"$\zeta$")
ax.legend()
plt.close()
fig.savefig(f"{figures}/model-interface_width.png", dpi = DPI)

In [ ]:
fig

From the plot, a good match between the critical expansion result for $\zeta$ and the simulated values are obtained at low $\chi/T$. At low $\chi/T$ values, the theory over-estimates the interface width. The lower than expected interface width is attributed to the small interface width being affected by finite size effects from discretization on the lattice.

In [ ]:
# ar = 1.25
# fig, ax = plt.subplots(1, 1, figsize = (figheight*ar, figheight), layout = "constrained")
# labels = ["Theory", "Simulation", "Critical expansion"]
# idx = 0

# # chi_theory = np.linspace(float(chi_real[0]), float(chi_real[-1]), 101)
# chi_theory = np.array(chi_real, dtype = float)
# sigma_theory = np.zeros_like(chi_theory)

# for i, chi in enumerate(chi_theory):
#     sigma_theory[i] = swift_theoretical_sigma(chi, kappa)

# ax.plot(chi_theory, sigma_theory, label = labels[idx], color = colors[idx], ls = linestyles[idx], lw = linewidth)
# idx += 1

# ax.plot(chi_real.astype(float), surface_tensions, label = labels[idx], color = colors[idx], ls = linestyles[idx], lw = linewidth, marker = markers[idx], markerfacecolor = "None", ms = markersize)
# idx += 1

# sigma_crit = (8*np.sqrt(kappa)/(chi_theory*T/2))*np.power((chi_theory*T/2 - T)/2, 1.5)
# ax.plot(chi_theory, sigma_crit, label = labels[idx], color = colors[idx], ls = linestyles[idx], marker = "None", markerfacecolor = "None", ms = markersize)
# idx += 1


# # ax.set_xlim([chi_theory[0], 2.25])
# ax.set_xlabel(r"$\frac{\chi}{T}$")
# ax.set_ylabel(r"$\sigma$")
# ax.legend()
# ax.set_title("Comparing theory and simulated "+r"$\sigma$"+" at " + f"$\kappa$={kappa}")
# # ax.set_yscale("log")
# # fig.savefig(f"{figures}/model-surface_tension-profile.png", dpi = DPI)

# Validation

## Homogenous mixture

A benchmark test to verify our implementation is to calculate the equilibration ratios of the LB modes to verify that the noise generated is consistent with the expected equilibrium values as defined in the $G$ matrix. To accomplish this, we perform simulations with spatially uncorrelated and correlated noise. Our system is $64^3$ with thermodynamic parameters $\chi = 0.4$, $T = 0.28$ and $\kappa = 0.01$. The relaxation rates are $\tau_{\rho} = 0.788$ and $\tau_{\psi} = 1$ and the fluctuation temperature is set to $k_B T = 10^{-7}$. We calculate the time correlation of each LB mode in k space, $\langle | \delta m (k) |^2 \rangle$ at each time-step. Results presented represent the average of 5000 simulation snapshots. We focus our attention on the conserved LB modes, $\delta \rho$, $\delta \phi$, $\delta \mathbf{u}$. 

\begin{figure}
    \centering
    \includegraphics[scale = 0.5]{figures/equilibration_ratios-spatially_independent.png}
    \caption{Spatially uncorrelated noise utilized in conjunction with a multi-component lattice boltzmann method based on a thermodynamic model used in Swift et al. 1996 with parameters, $\chi = 0.4$, $T = 0.28$, $\kappa = 0.01$, $\tau_{\rho} = 0.788$, $\tau_{\phi} = 1.0$. Plots (a), (b) and (c) demonstrate the equilibration ratios averaged at each shell of $k$ for the conserved moments. Plots (d), (e) and (f) demonstrate the angular dependence of the conserved LB moments at various $k$ values.}
    \label{fig:homogenous_spatially_independent}
\end{figure}

Figure \ref{fig:homogenous_spatially_independent} shows the equilibration ratios when using spatially uncorrelated noise. As reported by previous works, spatially uncorrelated noise results in unacceptably large deviations at higher wave numbers for $\rho$ and $\phi$. Compared to the single component multiphase case, the deviations observed are not as strong as there is no density gradient. While the equilibration ratios show that deviations from theory occur at high $k$ values, we see that the noise applied is isotropic 
This demonstrates that the noise covariance matrix even in its spatially uncorrelated form describes the fluctuation dissipation theorem for our multi component system appropriately at smaller $k$ values. To remove the deviations from theory at higher $k$ values, we also implement a spatially correlated $\Xi_k$ matrix and plot the results obtained from the same run plan used to generate Figure \ref{fig:homogenous_spatially_independent} in \ref{fig:homogenous_spatially_dependent}.

\begin{figure}
    \centering
    \includegraphics[scale = 0.5]{figures/equilibration_ratios-spatially_dependent.png}
    \caption{Spatially correlated noise utilized in conjunction with a multi-component lattice boltzmann method based on a thermodynamic model used in Swift et al. 1996 with parameters, $\chi = 0.4$, $T = 0.28$, $\kappa = 0.01$, $\tau_{\rho} = 0.788$, $\tau_{\phi} = 1.0$. Plots (a), (b) and (c) demonstrate the equilibration ratios averaged at each shell of $k$ for the conserved moments. Plots (d), (e) and (f) demonstrate the angular dependence of the conserved LB moments at various $k$ values.}
    \label{fig:homogenous_spatially_dependent}
\end{figure}

When looking at the spatially averaged equilibration ratios in Figure \ref{fig:homogenous_spatially_dependent} we see upon application of a spatially correlated noise covariance matrix that the deviations at high wave numbers disappears. The maximum error observed is at most $1 \%$ in all cases. The deviations at low $k$ values are attributed to the system not being equilibrated sufficiently. These results suggest that once spatial correlations are accounted for upon noise generation, $\Xi$ describes the fluctuation disspation theorem correctly at all $k$ values. While different parameters sets were not tested, as long as $\Xi$ is semi-positive definite we expect the same qualitative behavior seen here. 

In [ ]:
L = 64
boxDims = np.array([L, L, L])

chi = 0.4
T = 0.28
kappa = 0.01
kbt = 1e-7

rho0 = 1.0
phi0 = 0.0
thermo_vars = swift_et_al_1996_thermodynamic_model(rho0, phi0, chi, T, kappa)

binsize_avg = 16
binsize_radial = 16

freqs = np.fft.fftshift(np.fft.fftfreq(L))

In [ ]:
savedir = "."
T = 0.06
chi = 0.1
kappa = 0.01
thermo_vars = swift_et_al_1996_thermodynamic_model(1, 0, chi, T, kappa)
kbt = 1e-7
L = 32
binsize_avg = L//2
boxDim = [L, L, L]


ts = yt.load(f"{savedir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()
structure_factors = read_amrex_data(ds, boxDim)

ts = yt.load(f"{savedir}/hydro_plt*")
ds = ts[-1]
profile = read_amrex_data(ds, boxDim)
# profile.keys()

In [ ]:
plt.imshow(profile["phi"][L//2])
plt.colorbar()

In [ ]:
data = [structure_factors['struct_fact_rho_rho']/kbt]

slc = np.s_[L//2, L//2, L//2]

ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\rho$"]

rho_scale = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = 1, func = rho_scale)
        y[0] = np.NaN
        x, y = make_bins(x, binsize_avg, to_bin2 = y)
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
# ax.set_ylim([0.93, 1.07])
fig.tight_layout()
plt.close()
fig

In [ ]:
data = [structure_factors['struct_fact_phi_phi']/kbt]

ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\phi$"]

rho_func = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = scale_factor, func = rho_func, cs = False)
        x, y = make_bins(x, binsize_avg, to_bin2 = y)

        # y[0] = 1
        y[0] = np.NaN
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
fig.tight_layout()
plt.close()
fig

### Spatially Independent

#### Density

In [ ]:
noise_type = "spatially_independent"
save_dir = f"validation/equilibration_tests/{noise_type}"

ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

data_rho = np.array([read_amrex_data(ds, boxDims)['struct_fact_rho_rho']])
data_rho /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
ds.field_list

In [ ]:
data = data_rho
experiment_muck = thermo_vars.cs2k(kx, ky, kz)

test = data[0].copy()
test = np.fft.fftshift(test)

fig, axs = plt.subplots(1, 3, figsize = (9, 3))

ax = axs[0]
test *= (experiment_muck)

im = ax.imshow(test[L//2, :, :])
plt.colorbar(im, ax = ax, shrink = 0.8)

ax = axs[1]
im = ax.imshow(test[:, L//2, :])
plt.colorbar(im, ax = ax, shrink = 0.8)

ax = axs[2]
im = ax.imshow(test[:, :, L//2])
plt.colorbar(im, ax = ax, shrink = 0.8)

fig.tight_layout()
plt.close()

In [ ]:
slc = np.s_[L//2, L//2, L//2]

ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\rho$"]

rho_scale = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = 1, func = rho_scale)
        # y[0] = 1
        y[0] = np.NaN
        x, y = make_bins(x, binsize_avg, to_bin2 = y)
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.set_ylim([0.93, 1.07])
fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/rho-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1.5, 2.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax
    currData = d.copy()
    currData = np.fft.fftshift(currData)
    for j, r in enumerate(radii):
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r, func=rho_scale, scale_factor = scale_factor)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)


    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", fontsize = 14)
    ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", va = "bottom", ha = 'right', fontsize = 14, rotation = 'horizontal')
    ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/rho-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/rho-ER-{noise_type}-azimuth.png", dpi = 300)

#### Order parameter

In [ ]:
noise_type = "spatially_independent"
save_dir = f"validation/equilibration_tests/{noise_type}"
ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

data_phi = np.array([read_amrex_data(ds, boxDims)['struct_fact_phi_phi']])
data_phi /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
data = data_phi

experiment_muck = thermo_vars.mu_ck(kx, ky, kz)

test = data[0].copy()
test = np.fft.fftshift(test)
fig, axs = plt.subplots(1, 3, figsize = (9, 3))

ax = axs[0]
test *= (experiment_muck)
im = ax.imshow(test[L//2, :, :])
plt.colorbar(im, ax = ax)

ax = axs[1]
im = ax.imshow(test[:, L//2, :])
plt.colorbar(im, ax = ax)

ax = axs[2]
im = ax.imshow(test[:, :, L//2])
plt.colorbar(im, ax = ax)

fig.tight_layout()
plt.close()

In [ ]:
ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\phi$"]

rho_func = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = scale_factor, func = rho_func, cs = False)
        x, y = make_bins(x, binsize_avg, to_bin2 = y)

        # y[0] = 1
        y[0] = np.NaN
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
# print(x, y)
# ax.set_ylim([0.9, 1.1])
ax.set_ylim([0.93, 1.07])
fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/phi-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1.5, 2.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax
    currData = d.copy()
    currData = np.fft.fftshift(currData)
    for j, r in enumerate(radii):
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r, func=rho_scale, scale_factor = scale_factor, cs = False)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)


    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", fontsize = 14)
    ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", va = "bottom", ha = 'right', fontsize = 14, rotation = 'horizontal')
    ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/phi-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/phi-ER-{noise_type}-azimuth.png", dpi = 300)

#### Velocity

In [ ]:
noise_type = "spatially_independent"
save_dir = f"validation/equilibration_tests/{noise_type}"
ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

keys = ['struct_fact_ux_ux', 'struct_fact_uy_uy', 'struct_fact_uz_uz']

data_in = read_amrex_data(ds, boxDims)
data_v = np.empty((3, *boxDims))

for i, key in enumerate(keys):
    data_v[i] = data_in[key]

data_v /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
data = data_v
slc_s = [np.s_[L//2, :, :], np.s_[:, L//2, :], np.s_[:, :, L//2]]
labels = ['yz', 'xz', 'xy']
fig, axs = plt.subplots(3, 3, figsize = (9, 9))

for i in range(3):
    d = data[i].copy()
    d = np.fft.fftshift(d)
    for j in range(3):
        ax = axs[i, j]
        im = ax.imshow(d[slc_s[j]], vmin = 0.8, vmax = 1.2)
        plt.colorbar(im, ax = ax, shrink = 0.8)
        ax.set_title(f"$S_{{u_{chr(120+i)}u_{chr(120+i)}}}$ {labels[j]}")

fig.tight_layout()
plt.close()

In [ ]:
ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))
binsize = 8

labels = [r"$u_x$", r"$u_y$", r"$u_z$"]

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)

        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = 1)
        y[0] = np.NaN

        x, y = make_bins(x, binsize_avg, to_bin2 = y)
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

# ax.plot(x, [1]*x.size, 'lime', lw = 2)
ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | u_{\alpha}(k) | ^2 \rangle }{\rho_{0}k_b T}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.set_ylim([0.93, 1.07])

fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/vel-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
# fig1, tax = plt.subplots(1, 3, figsize = (sz*3, sz), subplot_kw={'projection': 'polar'})
# fig2, pax = plt.subplots(1, 3, figsize = (sz*3, sz), subplot_kw={'projection': 'polar'})
data = data_v[:1]
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1, 1.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax

    currData = d.copy()
    currData = np.fft.fftshift(currData)
    
    for j, r in enumerate(radii):
        # t, p, sf = radial_equilibration(d, rho0, phi0, radius = r)
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | u_{x} | ^2 \rangle }{\rho_0 k_B T}$", fontsize = 14)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | u_{x} | ^2 \rangle }{\rho_0 k_B T}$", fontsize = 14, ha = 'right', va = "top", rotation = 'horizontal')
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/vel-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/vel-ER-{noise_type}-azimuth.png", dpi = 300)

#### Figure compositing

In [ ]:
# independent composited figure
# %%capture
fig, axs = plt.subplots(2, 3, figsize=(3*figheight, 2*figheight))

noise_type = "spatially_independent"
ordering = ['rho', 'phi', 'vel']

downsize = 2
for i, ax in enumerate(axs[0]):
    pic_path = f"{figures}/{ordering[i]}-ER-{noise_type}-avg.png"
    img = imread(pic_path)
    if downsize is not None:
        img = resize(img, (img.shape[0]//downsize, img.shape[1]//downsize), anti_aliasing=True)
    
    ax.imshow(img)
    ax.axis('off')
    ax.text(0.0, 0.9, f"({chr(97+i)})", transform=ax.transAxes)

paths = [f"{figures}/rho-ER-{noise_type}-polar.png", f"{figures}/phi-ER-{noise_type}-polar.png", f"{figures}/vel-ER-{noise_type}-polar.png"]

for i, ax in enumerate(axs[1]):
    pic_path = paths[i]
    # print(pic_path)
    img = imread(pic_path)
    if downsize is not None:
        img = resize(img, (img.shape[0]//downsize, img.shape[1]//downsize), anti_aliasing=True)
    
    ax.imshow(img)
    ax.axis('off')
    ax.text(0.0, 0.9, f"({chr(97+3+i)})", transform=ax.transAxes)

fig.subplots_adjust(wspace=-0.1, hspace=0)
fig.tight_layout()
plt.close()
fig.savefig(f"{figures}/equilibration_ratios-{noise_type}.png", dpi=DPI)#, bbox_inches='tight')

### Spatially dependent

#### Density

In [ ]:
noise_type = "spatially_dependent"
save_dir = f"validation/equilibration_tests/{noise_type}"
ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

keys = ['struct_fact_rho_rho']
data_in = read_amrex_data(ds, boxDims)
data_rho = np.empty((len(keys), *boxDims))

for i, key in enumerate(keys):
    data_rho[i] = data_in[key]

data_rho /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
data = data_rho
experiment_muck = thermo_vars.cs2k(kx, ky, kz)

test = data[0].copy()
test = np.fft.fftshift(test)
# test /= kbt

fig, axs = plt.subplots(1, 3, figsize = (9, 3))

ax = axs[0]
test *= (experiment_muck)
# test[slc] /= test.sum()
im = ax.imshow(test[L//2, :, :])
plt.colorbar(im, ax = ax, shrink = 0.8)

ax = axs[1]
im = ax.imshow(test[:, L//2, :])
plt.colorbar(im, ax = ax, shrink = 0.8)

ax = axs[2]
im = ax.imshow(test[:, :, L//2])
plt.colorbar(im, ax = ax, shrink = 0.8)

fig.tight_layout()
plt.close()

In [ ]:
slc = np.s_[L//2, L//2, L//2]

ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\rho$"]

rho_scale = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = 1, func = rho_scale)
        y[0] = np.nan
        x, y = make_bins(x, binsize_avg, to_bin2 = y)
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

# ax.plot(x, [1]*x.size, 'lime', lw = 2)
ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.set_ylim([0.93, 1.07])
fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/rho-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1.5, 2.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax
    currData = d.copy()
    currData = np.fft.fftshift(currData)
    
    for j, r in enumerate(radii):
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r, func=rho_scale, scale_factor = scale_factor)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)


    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", fontsize = 14)
    ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | \rho(k) | ^2 \rangle }{S(k)}$", va = "bottom", ha = 'right', fontsize = 14, rotation = 'horizontal')
    ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/rho-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/rho-ER-{noise_type}-azimuth.png", dpi = 300)

#### Order parameter

In [ ]:
noise_type = "spatially_dependent"
save_dir = f"validation/equilibration_tests/{noise_type}"
ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

keys = ['struct_fact_phi_phi']
data_phi = np.empty((len(keys), *boxDims))
data_in = read_amrex_data(ds, boxDims)
for i, key in enumerate(keys):
    data_phi[i] = data_in[key]

data_phi /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
data = data_phi

experiment_muck = thermo_vars.mu_ck(kx, ky, kz)

test = data[0].copy()
test = np.fft.fftshift(test)

fig, axs = plt.subplots(1, 3, figsize = (9, 3))

ax = axs[0]
test *= (experiment_muck)
im = ax.imshow(test[L//2, :, :])
plt.colorbar(im, ax = ax)

ax = axs[1]
im = ax.imshow(test[:, L//2, :])
plt.colorbar(im, ax = ax)

ax = axs[2]
im = ax.imshow(test[:, :, L//2])
plt.colorbar(im, ax = ax)

fig.tight_layout()
plt.close()

In [ ]:
ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$\phi$"]

rho_func = lambda a, c: a*c
scale_factor = 1

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = scale_factor, func = rho_func, cs = False)
        x, y = make_bins(x, binsize_avg, to_bin2 = y)
        y[0] = np.nan
        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

# ax.plot(x, [1]*x.size, 'lime', lw = 2)
ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
# print(x, y)
ax.set_ylim([0.93, 1.07])
fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/phi-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1.5, 2.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax
    currData = d.copy()
    currData = np.fft.fftshift(currData)
    for j, r in enumerate(radii):
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r, func=rho_scale, scale_factor = scale_factor, cs = False)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)


    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", fontsize = 14)
    ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | \phi(k) | ^2 \rangle }{S(k)}$", va = "bottom", ha = 'right', fontsize = 14, rotation = 'horizontal')
    ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/phi-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/phi-ER-{noise_type}-azimuth.png", dpi = 300)

#### Velocities

In [ ]:
noise_type = "spatially_dependent"
save_dir = f"validation/equilibration_tests/{noise_type}"
ts = yt.load(f"{save_dir}/SF_plt_mag*")
ds = ts[-1]
ds.force_periodicity()

keys = ['struct_fact_ux_ux', 'struct_fact_uy_uy', 'struct_fact_uz_uz']
data_v = np.empty((len(keys), *boxDims))
data_in = read_amrex_data(ds, boxDims)
for i, key in enumerate(keys):
    data_v[i] = data_in[key]
data_v /= kbt

if "in" in noise_type:
    kx, ky, kz = [0, 0, 0]
else:
    kx, ky, kz = np.meshgrid(*tuple([2*np.pi*freqs for L in [L, L, L]]), indexing='ij')

In [ ]:
data = data_v
slc_s = [np.s_[L//2, :, :], np.s_[:, L//2, :], np.s_[:, :, L//2]]
labels = ['yz', 'xz', 'xy']
fig, axs = plt.subplots(3, 3, figsize = (9, 9))

for i in range(3):
    d = data[i].copy()
    for j in range(3):
        ax = axs[i, j]
        im = ax.imshow(d[slc_s[j]], vmin = 0.8, vmax = 1.2)
        plt.colorbar(im, ax = ax, shrink = 0.8)
        ax.set_title(f"$S_{{u_{chr(120+i)}u_{chr(120+i)}}}$ {labels[j]}")

fig.tight_layout()
plt.close()

In [ ]:
ar = 1.25
sz = figheight
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

labels = [r"$u_x$", r"$u_y$", r"$u_z$"]

for i, d in enumerate(data):
        currData = d.copy()
        currData = np.fft.fftshift(currData)
        x, y = spherically_averaged_structure_factor(currData, thermo_vars, scale_factor = 1)
        y[0] = np.nan      
        x, y = make_bins(x, binsize_avg, to_bin2 = y)

        ax.plot(x, y, 
                marker = markers[i], color = colors[i],
                markersize = markersize, linestyle = "None", label = labels[i],
                markerfacecolor = "None")

ax.axhline(y = 1, xmin = 0.01, xmax = 0.99, color = "k", ls = "-", lw = linewidth)

ax.set_xlabel(r"$k$", fontsize = 14)
ax.set_ylabel(r"$\frac{ \langle | u_{\alpha}(k) | ^2 \rangle }{\rho_{0}k_b T}$", fontsize = 14)

ax.tick_params(axis='both', which='major', labelsize=12)
ax.tick_params(axis='both', which='minor', labelsize=10)
ax.legend()
ax.set_ylim([0.93, 1.07])

fig.tight_layout()
plt.close()
fig.savefig(f"./{figures}/vel-ER-{noise_type}-avg.png", dpi = 300)

In [ ]:
sz = figheight
# fig1, tax = plt.subplots(1, 3, figsize = (sz*3, sz), subplot_kw={'projection': 'polar'})
# fig2, pax = plt.subplots(1, 3, figsize = (sz*3, sz), subplot_kw={'projection': 'polar'})
data = data_v[:1]
fig1, tax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})
fig2, pax = plt.subplots(1, 1, figsize = (sz, sz), subplot_kw={'projection': 'polar'})

radii = [0.5, 1, 1.5]

t_angle = 0
p_angle = 0 

for i, d in enumerate(data):
    ax1 = tax
    ax2 = pax

    currData = d.copy()
    currData = np.fft.fftshift(currData)
    
    for j, r in enumerate(radii):
        t, p, sf = radial_equilibration(currData, thermo_vars, radius = r)

        idxs = np.isclose(np.abs(t), t_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(p)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax1.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

        idxs = np.isclose(np.abs(p), p_angle/180*np.pi, atol = np.pi/L)
        angles = np.abs(t)[idxs]
        segment = sf[idxs]
        angles, segment =  make_bins(angles, binsize_radial, to_bin2 = segment)
        ax2.plot(angles, segment, color = colors[j], label = f"R = {r}",lw = linewidth)

    ax1.set_rticks([0.5, 1.0])
    ax1.set_thetalim([0, np.pi/2])
    ax1.set_ylabel(r"$\frac{ \langle | u_{x} | ^2 \rangle }{\rho_0 k_B T}$", fontsize = 14)
    # ax1.set_title(r"$\theta = {{{0}}}^{{\circ}}$".format(t_angle))

    ax2.set_rticks([0.5, 1.0])
    ax2.set_thetalim([0, np.pi])
    ax2.set_ylabel(r"$\frac{ \langle | u_{x} | ^2 \rangle }{\rho_0 k_B T}$", fontsize = 14, ha = 'right', va = "top", rotation = 'horizontal')
    # ax2.set_title(r"$\psi = {{{0}}}^{{\circ}}$".format(p_angle))

ax1.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)
ax2.legend(loc = 'upper right', bbox_to_anchor=(1.1, 1.1), fancybox=True)

fig1.tight_layout()
fig2.tight_layout()
plt.close()
fig1.savefig(f"./{figures}/vel-ER-{noise_type}-polar.png", dpi = 300)
fig2.savefig(f"./{figures}/vel-ER-{noise_type}-azimuth.png", dpi = 300)

#### Figure compositing

In [ ]:
# dependent composited figure
# %%capture
fig, axs = plt.subplots(2, 3, figsize=(3*figheight, 2*figheight))

noise_type = "spatially_dependent"
ordering = ['rho', 'phi', 'vel']

downsize = None
for i, ax in enumerate(axs[0]):
    pic_path = f"{figures}/{ordering[i]}-ER-{noise_type}-avg.png"
    img = iio.imread(pic_path)
    if downsize is not None:
        img = ski.transform.resize(img, (img.shape[0]//downsize, img.shape[1]//downsize), anti_aliasing=True)
    
    ax.imshow(img)
    ax.axis('off')
    ax.text(0.0, 0.9, f"({chr(97+i)})", transform=ax.transAxes)

paths = [f"{figures}/rho-ER-{noise_type}-polar.png", f"{figures}/phi-ER-{noise_type}-polar.png", f"{figures}/vel-ER-{noise_type}-polar.png"]

for i, ax in enumerate(axs[1]):
    pic_path = paths[i]
    # print(pic_path)
    img = iio.imread(pic_path)
    if downsize is not None:
        img = ski.transform.resize(img, (img.shape[0]//downsize, img.shape[1]//downsize), anti_aliasing=True)
    
    ax.imshow(img)
    ax.axis('off')
    ax.text(0.0, 0.9, f"({chr(97+3+i)})", transform=ax.transAxes)

fig.subplots_adjust(wspace=-0.1, hspace=0)
fig.tight_layout()
plt.close()
fig.savefig(f"{figures}/equilibration_ratios-{noise_type}.png", dpi=DPI)#, bbox_inches='tight')

## Interface fluctuations

To validate the implementation, the equilibration ratios of the density, order parameter and velocity are calculated for a mixed system of equal volume fractions of each fluid. Next, the interfacial fluctuations are calculated using Eqaution 3.11 in calculated by [Grant and Desai 1983](https://journals.aps.org/pra/pdf/10.1103/PhysRevA.27.2577)

In [ ]:
nx = 64
ny = 1
nz = 512
boxDim = np.array([nx, ny, nz])

noise_type = "spatially_independent"
chi = 0.42
T = 0.2
kappa = 0.01
kbt = 1e-7

sigma_theory = swift_theoretical_sigma(chi/T, kappa)
# sigma_theory = swift_critical_sigma(1.0, chi, T, kappa)[0]
print(f"{sigma_theory:.3e}")

lc = np.sqrt(kappa/(2*(chi/T)))
l_star = np.sqrt(kbt/sigma_theory)
print(r"$l^{*}$="+ f"{l_star:.3e}")
print(r"$l_{c}$="+ f"{lc:.3e}")

# savedir = f"./validation/interface/spatially_independent"
savedir = f"./test/interface_fluctuation"
idx = -1
output_file = "hydro_plt"

fft_mode = "ortho"

In [ ]:
ts = yt.load(f"{savedir}/{output_file}*")
ds = ts[idx]
t = int(ds.current_time)
ad = ds.all_data()
data_in = read_amrex_data(ds, boxDim)

rho = data_in["rho"]
phi = data_in["phi"]
C1 = (rho+phi)/2

C0 = swift_theoretical_C0(chi/T, kappa)
phi0 = 2*C0 - 1
xi = swift_theoretical_xi(C0, chi/T, kappa)
fit_func = lambda x, b:-phi0*np.tanh((x - b)/(np.sqrt(2)*xi))

im_slc = np.s_[:, ny//2, :]
profile_slc = np.s_[nx//4:3*nx//4, ny//2 , nz//2]

# https://matplotlib.org/3.5.0/tutorials/intermediate/gridspec.html

sz = 4
widths = [sz]
heights = [sz//4, sz//4, sz//2]
fig = plt.figure(layout = "constrained", figsize = (20, 8))
gs0 = fig.add_gridspec(ncols = 1, nrows = 3, width_ratios = widths, height_ratios = heights)

ax = fig.add_subplot(gs0[0])
im = ax.imshow(rho[im_slc], cmap = 'cividis')
ax.set_xlabel("y")
ax.set_ylabel("x")
plt.colorbar(im, ax = ax, label = r"$\rho$", orientation = "horizontal")

ax = fig.add_subplot(gs0[1])
im = ax.imshow(phi[im_slc], cmap = 'cividis')
ax.set_xlabel("y")
ax.set_ylabel("x")
plt.colorbar(im, ax = ax, label = r"$\phi$", orientation = "horizontal")
ax2 = ax.twinx()

method = "direct"
h = np.mean(interface_height(phi, zero = True, method = method, thermodynamic_params=[chi, T, kappa]), axis = 0)
ax2.plot(h, label = method, color = "r", ls = "-")
ax2.legend()

# print(l_star/np.abs(h).max())
# print(lc/np.abs(h).max(), lc, np.abs(h).max())
print(f"lc:{lc:.3f}", f"h:{np.abs(h).max()}", f"lc ratio:{lc/np.abs(h).max():.3f}", f"l* ratio:{l_star/np.abs(h).max():.3f}")

gs1 = gs0[2].subgridspec(1, 2)
ax = fig.add_subplot(gs1[0])
x = np.arange(nx//4, 3*nx//4, 1)
im = ax.plot(x, rho[profile_slc], 'rx', label = "profile")
ax.plot(x, np.ones(x.size), 'ko', label = "reference", markerfacecolor="None")

ax.set_xlabel("x")
ax.set_ylabel(r"$\rho$")
ax.legend(ncol = 1, fontsize = 'small')

ax = fig.add_subplot(gs1[1])
x = np.arange(nx//4, 3*nx//4, 1)
im = ax.plot(x, phi[profile_slc], 'rx', label = "profile")

y = fit_func(x, (nx - 1)/2)
ax.plot(x, y, 'ko', label = "referece", markerfacecolor="None")

ax.set_xlabel("x")
ax.set_ylabel(r"$\phi$")
ax.legend(ncol = 1, fontsize = 'small')

plt.close()
fig.suptitle(f"$\kappa$ = {kappa}, $\chi = {chi}, T = {T}, t = {t}$")
fig

In [ ]:
heights_k_fft = np.zeros((nz), dtype = np.complex128)
k1_fft = np.fft.fftfreq(nz)*2*np.pi

data_paths = yt.load(f"{savedir}/{output_file}*")[1:]

for path in data_paths:
    profile = read_amrex_data(path, boxDim)
    phi = profile["phi"]
    rho = profile["rho"]
    C1 = 0.5*(rho + phi)

    # plot = phi
    # level = 0

    plot = C1
    level = 0.5

    # min_level = np.mean(plot[nx//4, :, :])
    # max_level = np.mean(plot[3*nx//4, :, :])
    # level = 0.5*(min_level + max_level)

    h = interface_height(plot, zero = False, method = "direct", thermodynamic_params=[chi, T, kappa], level = level)
    h = np.mean(h, axis = 0)

    h_k = np.fft.fft(h, norm = fft_mode)
    heights_k_fft += h_k*h_k.conjugate()

heights_k_fft = np.abs(heights_k_fft)
heights_k_fft /= len(data_paths)
plt.close()

In [ ]:
idx = 0
sz = figheight
ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (sz*ar, sz))

## SIMULATION RESULTS ##
xbin, ybin = bin_height_spectrum_1D(k1_fft, heights_k_fft, nz//4)

factor = 1
ax.loglog(xbin, factor*ybin, label = "Simulation", ms = 6, color = colors[idx], ls = "None", marker = markers[idx], markerfacecolor = "None")
idx += 1
## SIMULATION RESULTS ##

## THEORETICAL RESULTS ##
freqs_theory = np.linspace(0, np.pi, 1001)
interface_fluct_theory = kbt/(sigma_theory*np.power(freqs_theory, 2))
# interface_fluct_theory /= nz*ny

ax.loglog(freqs_theory, interface_fluct_theory, label = "Theory", color = colors[idx], linestyle = "-", lw = 2, marker = "None")
idx += 1
## THEORETICAL RESULTS ##

ax.set_xlabel(r"$k$", fontsize = 15)
ax.set_ylabel(r"$\langle |h(k)|^2 \rangle$", fontsize = 15)
ax.tick_params(axis='both', which='major', labelsize=12)
ax.legend(fontsize = 12, loc = 'lower left')

fig.tight_layout()
plt.close()
fig.savefig(f"{figures}/interface-height_flucuations-{noise_type}.png", dpi = 300)

In [ ]:
fig

Next, we benchmark our fluctuating lattice boltzmann method when the system is phase separated. We first verify that the fluctuating LBM system reproduces the correct noise spectrum for a phase separated system using a flat interface between two stripes of fluid. Capillary waves induced from thermal fluctuations will induce roughening of the interface, resulting in the position of the interface changing over time. Previous work has identified the capillary wave spectrum given by,

\begin{equation}
    \langle |h(\mathbf{k})|^2 \rangle = \frac{k_B T}{A \sigma k^2}
    \label{eq:capillary_wave_theory}
\end{equation}

In Eq \ref{eq:capillary_wave_theory}, $\sigma$, $k^2$ and $A$ represent the surface tension, linear wave number and interface area respectively. The box is filled with fluid such that the interface is parallel with the long axis of a pseudo 2D box with dimensions $(L_x, L_y, L_z) = (64, 1, 512)$. In previous work which utilized a multiphase fluid, the density contrast present in the system necessitated use of Galilean invariance correction terms. These correction terms are not necessary in our implementation as the total fluid density $\rho$ is constant. The interface height is defined as the point where $C_1 = 0.5$. As this point did not always explicitly exist in the data, an interpolation between points adjacent to the interface is conducted to calculate the interface height. We used the interpolation scheme as implemented in \texttt{skimage.measure.find\_contours}. We set the local reference values of $\rho$ and $\phi$ when computing $\Xi_{k \rightarrow 0}$ by equilibrating the interface before switching fluctuation on, after which the system was run for 300000 timesteps. The capillary spectrum as a function of the wave number $k$ is plotted in Figure \ref{fig:height_fluctuation_fig}.

\begin{figure}
    \centering
    \includegraphics[scale=0.5]{figures/interface-height_flucuations-spatially_independent.png}
    \caption{Height fluctuation spectrum of a planar interface obtained from $\Xi_{k \rightarrow 0}$ in a free energy multicomponent lattice boltzmann method. The line represents the theoretical prediction of the spectrum while the markers represent the results from our simulations. The thermodynamic model parameters are, $T = 0.2, \chi = 0.42, \kappa = 0.01$, $k_B T = 10^{-7}$.}
    \label{fig:height_fluctuation_fig}
\end{figure}

Figure \ref{fig:height_fluctuation_fig} shows that the simulation results match the theoretical height fluctuations calculated using Equation \ref{eq:capillary_wave_theory} for $k \leq 1$. The deviations from theory at low and high $k$ values are attributed to compressibility of the liquid stripe causing suppression of fluctuations at large wavelengths and the assumptions used to develop capillary wave theory breaking down at small length scales respectively. 

## Droplet shape fluctuations

Finally, the surface tension calculated from the fluctuations of the shape of a droplet will be compared to that calculated using the Young-Laplace equation as defined in [Benayad et al. 2020](https://doi.org/10.1021/acs.jctc.0c01064).

### L = 128

In [ ]:
# box size
nx = 128
ny = 128
nz = 128
boxDim = np.array([nx, ny, nz])

# thermodynamic information used to conduct simulations
chi_ratio = 2.1
T = 0.2
chi = chi_ratio*T
kappa = 0.01
kbt = 1e-7
variable = "C1"

# sigma_theory = swift_critical_sigma(1.0, chi, T, kappa)[0]
sigma_theory = swift_theoretical_sigma(chi_ratio, kappa)
C0_theory = swift_theoretical_C0(chi_ratio)
zeta_theory = swift_theoretical_xi(C0_theory, chi_ratio, kappa)

# slc = slice(100, 3000)
slc = slice(2000, 10000)
# slc = slice(10000, 20000)
# analysis_file = "analysis-150000.csv"
analysis_file = "analysis.csv.0"
window_size = 100
savedir = f"/scratch/FLB/validation/droplet_fluctuations"

In [ ]:
ds = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")[-1]
non_fluct_profile = read_amrex_data(ds, boxDim)
currData = non_fluct_profile["phi"]
plt.imshow(currData[nx//2])

In [ ]:
# def read_csv(path):
#     ls = []
#     with open(path) as csvfile:
#         reader = csv.reader(csvfile, delimiter=" ", quotechar="|")
#         for row in reader:
#             try:
#                 ls.append([float(i) for i in row[0].split(",")[:-1]])
#             except ValueError:
#                 ls.append([i for i in row[0].split(",")[:-1]])
#         ls = np.array(ls)
#     return ls

In [ ]:
data = read_csv(f"{savedir}/non_fluctuating/{analysis_file}")
data = data[1:, :].astype(float)

timesteps = data[:, 0]
Rs = data[:, 1]

idx = 0
plt.plot(timesteps, Rs, label = "Simulation",
         color = colors[idx], ls = linestyles[idx],
         marker = markers[idx], markerfacecolor = "None", ms = markersize)

In [ ]:
to_plot = non_fluct_profile

rho = to_plot["rho"]
phi = to_plot["phi"]

analyze = phi
level = 0

# analyze = (rho+phi)/2
# level = 0.5

verts, _, _, _ = measure.marching_cubes(analyze, level = level, step_size = 1)

center = verts.mean(axis = 0)
shifted = verts - center[np.newaxis, :]

x, y, z = shifted[:, 0], shifted[:, 1], shifted[:, 2]

r = np.sqrt(x**2 + y**2 + z**2)
theta = np.arccos(z / r)
phi = np.arctan2(y, x)
phi[phi < 0] += 2 * np.pi

shifted_spherical = np.array([r, theta, phi]).T

from scipy.special import sph_harm_y
# from scipy.linalg import lstsq
from numpy.linalg import lstsq

Lmax = 8
cols = np.sum(2*np.arange(0, Lmax+1, 1)+1)
rows = shifted_spherical.shape[0]

A = np.empty((rows, cols), dtype = complex)

curr_col = 0
d_zeta_lm = {}
for l in range(Lmax+1):
    for m in range(-l, l+1, 1):
        d_zeta_lm[f"{l, m}"] = 0
        for n_curr_vertex in range(cols):
            theta = shifted_spherical[n_curr_vertex, 1]
            phi = shifted_spherical[n_curr_vertex, 2]
            A[n_curr_vertex, curr_col] = sph_harm_y(l, m, theta, phi)
        curr_col += 1

f = shifted_spherical[:, 0] - shifted_spherical[:, 0].mean()
f = f.astype(complex)
# c, residuals, rank, s = lstsq(A, f, rcond=None)
U, s, Vh = np.linalg.svd(A, full_matrices=False)
s_inv = np.diag(1 / s)
c = Vh.T @ s_inv @ U.T @ f

zeta_lm_mag = np.abs(c**2).astype(float)

curr_col = 0
for l in range(Lmax+1):
    for m in range(-l, l+1, 1):
        d_zeta_lm[f"{l, m}"] = zeta_lm_mag[curr_col]
        curr_col += 1


d_zeta_lm

In [ ]:
curr_density = (non_fluct_profile["phi"] + non_fluct_profile["rho"])/2
level = 0.5

# curr_density = non_fluct_profile["phi"]
# level = 0

plt.imshow(curr_density[nx//2])
plt.colorbar()

out1, out2, out3 = spherical_harmonic_expansion(curr_density, level = level, lmax = 4, debug = True)
np.abs(out1**2).astype(float)

### Amrex scripts

In [ ]:
# box size
nx = 64
ny = 64
nz = 64
boxDim = np.array([nx, ny, nz])

# thermodynamic information used to conduct simulations
chi_ratio = 2.1
T = 0.1
chi = chi_ratio*T
kappa = 0.005
kbt = 1e-7
variable = "C1"

sigma_theory = swift_critical_sigma(1.0, chi, T, kappa)[0]
# sigma_theory = swift_theoretical_sigma(chi_ratio, kappa)
C0_theory = swift_theoretical_C0(chi_ratio)
zeta_theory = swift_theoretical_xi(C0_theory, chi_ratio, kappa)

# slc = slice(100, 3000)
slc = slice(2000, 10000)
# slc = slice(10000, 20000)
# analysis_file = "analysis-150000.csv"
analysis_file = "analysis.csv.0"
window_size = 100
# savedir = f"/scratch/FLB/validation/droplet_fluctuations"
savedir = f"validation/droplet_fluctuations/T_{T}/kappa_{kappa}/"
# savedir = f"validation/droplet_fluctuations/T_{T}/kappa_{kappa}/fix_ref"
# savedir = f"validation/droplet_fluctuations/T_{T}/kappa_{kappa}/add_cs2"
# savedir = f"validation/droplet_fluctuations/spatially_independent/kappa_{kappa}/"

In [ ]:
ds = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")[-1]
# ds.force_periodicity()
non_fluct_profile = read_amrex_data(ds, boxDim)
currData = non_fluct_profile["phi"]
# currData = np.fft.fftshift(currData)
plt.imshow(currData[nx//2])

In [ ]:
R_det = 0

ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

if os.path.exists(f"{savedir}/{analysis_file}"):
    fluctuating_data = read_csv(f"{savedir}/{analysis_file}")
    x = fluctuating_data[slc, 0]
    # x -= x[0]
    y = fluctuating_data[slc, 1]

    ax.plot(x, y, label = "R",
            color = colors[0])
    
    x_window = moving_average(x, window_size=window_size)
    y_window = moving_average(y, window_size=window_size)
    
    ax.plot(x_window, y_window, label = r"$R^{window}$",
            color = colors[1])

    ax.axhline(y.mean(), label = r"$\langle R \rangle$",
               color = "magenta")

ds = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")[-1]
non_fluct_profile = read_amrex_data(ds, boxDim)
if variable == "C1":
    non_fluct_to_plot = (non_fluct_profile["rho"] + non_fluct_profile["phi"])/2
    level = 0.5
else:
    non_fluct_to_plot = non_fluct_profile["phi"]
    level = 0

R_det = droplet_radius_mass(np.where(non_fluct_to_plot > level, non_fluct_to_plot, 0))
ax.axhline(R_det, label = r"$R_{det}$",
           color = "lime")

young_laplace_sigma = R_det*pressure_jump(non_fluct_profile["rho"]*(T + cs2_ideal))/2

ax.legend()
ax.set_ylabel(r"$R_d \Delta x$")
ax.set_xlabel(r"$t \Delta t$")
plt.close()
fig

In [ ]:
dR = fluctuating_data[slc, -3:]
R = fluctuating_data[slc, 1]

ar = 1
fig, axs = plt.subplots(1, 2, figsize = (ar*figheight*2, figheight))
alpha = 0.7
nbin = 20
edgecolor = "None"

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, 
                              edgecolor = edgecolor, bins = nbin, 
                              color = colors[0], label = "R")

ax.axvline(np.mean(R), color = "magenta", 
           ls = "--", lw = 1, label = r"$\langle R \rangle$")

ax.axvline(R_det, color = "lime", 
           ls = "--", lw = 1, label = r"$R_{det}$")

ax.set_title("Radius fluctuation")
ax.legend()

ax = axs[1]

dR2 = dR.copy()
dR2 = np.sqrt(dR2)
dR2 -= 1.0
dR2 *= R.mean()

for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                  label = f"$\delta${chr(i+97)}",
                                  alpha = alpha, edgecolor = edgecolor, 
                                  bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
print(f"sigma_f:{np.mean(sigma_fluct):.4e}, sigma_t:{sigma_theory:.4e}, sigma_yp:{young_laplace_sigma:.4e}, delta sigma_f:{np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory):.4f}")
plt.close()
print(sigma_fluct)
fig
# fig.savefig(f"{figures}/droplet-shape_flucuations-{noise_type}.png", dpi = 300)


In [ ]:
import uschille_droplet_scripts as uschill_analysis

ts = yt.load(f"{savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
zeta = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for i, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[i] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # filtered = to_plot
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    # coms_all[i] = com
    # dR[i] = axial_radii(filtered, com)
    # gr = gyration_tensor(com, filtered) # Calculating the gyration tensor of the droplet
    # egr = np.sqrt(np.linalg.eigvals(gr)) # calculating the unordered eigenvalues of the gyration tensor
    
    # curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    # curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', binned = False)
    # R[i] = curr_fit[0]
    # zeta[i] = curr_fit[-1]
    # R[i] = droplet_radius_mass(filtered)

    com = uschill_analysis.center_of_mass(filtered)
    S = uschill_analysis.gyration_tensor(filtered, cm=com)
    egr = np.sqrt(np.linalg.eigvals(S)) # calculating the unordered eigenvalues of the gyration tensor
    da = np.power(egr[0], 1/3)/np.power(np.prod(egr[[1,2]]), 1/6) # calculating the variations in dx
    db = np.power(egr[1], 1/3)/np.power(np.prod(egr[[0,2]]), 1/6) # calculating the variations in dy
    dc = np.power(egr[2], 1/3)/np.power(np.prod(egr[[0,1]]), 1/6) # calculating the variations in dz 

    dR[i] = [da, db, dc]

    curr_fit = uschill_analysis.droplet_profile(filtered) #R, alpha, rhoh, rhol = p
    R[i] = curr_fit[0]
    zeta[i] = curr_fit[1]

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (ar*figheight*2, figheight))
alpha = 0.7
nbin = 20
edgecolor = "None"

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, 
                              edgecolor = edgecolor, bins = nbin, 
                              color = colors[0], label = "R")

ax.axvline(R.mean(), color = "magenta", 
           ls = "--", lw = 1, label = r"$\langle R \rangle$")

ds = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")[-1]
non_fluct_profile = read_amrex_data(ds, boxDim)
if variable == "C1":
    non_fluct_to_plot = (non_fluct_profile["rho"] + non_fluct_profile["phi"])/2
    level = 0.5
else:
    non_fluct_to_plot = non_fluct_profile["phi"]
    level = 0

# R_det = droplet_radius_mass(np.where(non_fluct_to_plot > level, non_fluct_to_plot, 0))
curr_fit = uschill_analysis.droplet_profile(np.where(non_fluct_to_plot > level, non_fluct_to_plot, 0)) #R, alpha, rhoh, rhol = p

ax.axvline(curr_fit[0], color = "lime", 
           ls = "--", lw = 1, label = r"$R_{det}$")

ax.set_title("Radius fluctuation")
ax.legend()

ax = axs[1]

dR2 = dR.copy()
# dR2 = np.sqrt(dR2)
dR2 -= 1.0
dR2 *= R.mean()

for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                  label = f"$\delta${chr(i+97)}",
                                  alpha = alpha, edgecolor = edgecolor, 
                                  bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
plt.close()
print(sigma_fluct)
fig
# fig.savefig(f"{figures}/droplet-shape_flucuations-{noise_type}.png", dpi = 300)


In [ ]:
ts = yt.load(f"{savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
zeta = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for i, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[i] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # filtered = to_plot
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[i] = com
    dR[i] = axial_radii(filtered, com)
    
    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'erf', binned = False)
    R[i] = curr_fit[0]
    zeta[i] = curr_fit[-1]
    R[i] = droplet_radius_mass(filtered)

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (ar*figheight*2, figheight))
alpha = 0.7
nbin = 20
edgecolor = "None"

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, 
                              edgecolor = edgecolor, bins = nbin, 
                              color = colors[0], label = "R")

ax.axvline(R.mean(), color = "magenta", 
           ls = "--", lw = 1, label = r"$\langle R \rangle$")

# ax.axvline(R_det, color = "lime", 
        #    ls = "--", lw = 1, label = r"$R_{det}$")

ax.set_title("Radius fluctuation")
ax.legend()

ax = axs[1]

dR2 = dR.copy()
# dR2 = np.sqrt(dR2)
dR2 -= 1.0
dR2 *= R.mean()

for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                  label = f"$\delta${chr(i+97)}",
                                  alpha = alpha, edgecolor = edgecolor, 
                                  bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
plt.close()
print(sigma_fluct)
fig
# fig.savefig(f"{figures}/droplet-shape_flucuations-{noise_type}.png", dpi = 300)


In [ ]:
ts = yt.load(f"{savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
lmax = 4
zeta_lm = np.zeros(np.sum(2*np.arange(0, lmax+1, 1)+1), dtype = complex)

for i, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[i] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    filtered = to_plot
    # filtered = np.where(to_plot > level, to_plot, 0)

    curr_zeta_lm = spherical_harmonic_expansion(filtered, level = level, lmax = lmax, debug = False)
    zeta_lm += np.abs(curr_zeta_lm**2)

In [ ]:
curr_zeta_lm

In [ ]:
zeta_lm

In [ ]:
profile = read_amrex_data(yt.load(f"{savedir}/non_fluctuating/hydro_plt*")[-1], boxDim)
if variable == "phi":
    non_fluct_to_plot = profile['phi']
else:
    non_fluct_to_plot = (profile['phi'] + profile['rho'])/2

non_fluctuating_fit, _, _ = droplet_radius_profile_1d(non_fluct_to_plot, binned = False)
zeta_non_fluctuating = non_fluctuating_fit[-1]

print(f"zeta_sim:{zeta_non_fluctuating:.4e}, zeta_fluct:{zeta.mean():.4e}, delta_zeta:{zeta_non_fluctuating-zeta.mean():.4e}")

In [ ]:
fit_params_fluct, radii_fluct, profile_fluct = droplet_radius_profile_1d(filtered, com, binned = False)
fit_params_det, radii_det, profile_det = droplet_radius_profile_1d(non_fluct_to_plot, binned = False)

ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

radii = np.arange(fit_params_det[0]-5*fit_params_det[-1], fit_params_det[0]+5*fit_params_det[-1], 0.1)

profile_fit = lambda x, R, hi, lo, iw:lo - hi*np.tanh((x - R)/(np.sqrt(2)*iw))

ax.plot(radii, profile_fit(radii, *fit_params_fluct),
        color = colors[0], label = r"$C1^{fluct}(r)$")
ax.axvline(fit_params_fluct[0], ymin = 0, ymax = 1,
           color = colors[0])

ax.plot(radii, profile_fit(radii, *fit_params_det),
        color = colors[1], label = r"$C1^{det}(r)$")
ax.axvline(fit_params_det[0], ymin = 0, ymax = 1,
           color = colors[1])

ax.legend()

plt.close()
fig

In [ ]:
zeta_non_fluctuating - zeta.mean()

### Python scripts

In [ ]:
# Details box dimensions and information location (long run and big system)
nx = 64
ny = 64
nz = 64
boxDim = np.array([nx, ny, nz])
nbins = boxDim.min()

# thermodynamic information used to conduct simulations
chi = 0.42
T = 0.2
kappa = 0.01
kbt = 1e-7
noise_type = "spatially_independent"
variable = "C1"

# sigma_theory = swift_critical_sigma(1.0, chi, T, kappa)
sigma_theory = swift_theoretical_sigma(chi/T, kappa)
C0 = swift_theoretical_C0(chi/T)
zeta = swift_theoretical_xi(C0, chi/T, kappa)

idx = -1

# savedir = f"./validation/droplet_fluctuations/spatially_independent/kappa_{kappa}/"
savedir = f"./validation/droplet_fluctuations/T_{T}//kappa_{kappa}"
filename = "fluctuating_on"

output_file = "hydro_plt"
center_slc = np.s_[nx//2-1:nx//2+2, ny//2-1:ny//2+2, nz//2-1:nz//2+2]
edge_slc = np.s_[0:nx:nx-1, 0:ny:ny-1, 0:nz:nz-1]

In [ ]:
# visualizing the droplet in the system
ts = yt.load(f"{savedir}/{output_file}*")
ds = ts[idx]
ad = ds.all_data()
profile = read_amrex_data(ds, boxDim)

rho = profile["rho"]
phi = profile["phi"]
C1 = (rho+phi)/2
C0 = swift_theoretical_C0(chi/T)

# level = np.mean(phi[center_slc]) + np.mean(phi[edge_slc])
level = (C0 + 1 - C0)/2
# filt = sharpen_droplet_interface(C1, int_height = level, min_val = 1 - C0, max_val = C0, dcf = 1/5)
# print(1 - C0)
t = int(ds.current_time)

fig, axs = plt.subplots(2, 2, figsize = (8, 8))
# axs = axs.flatten()

ax = axs[0, 0]
im = ax.imshow(rho[:, : , nz//2])
ax.set_xlabel("y")
ax.set_ylabel("x")
plt.colorbar(im, ax = ax, label = r"$\rho$")

ax = axs[0, 1]
x = np.arange(0, nx, 1)
im = ax.plot(x, rho[:, ny//2 , nz//2], 'rx', label = "profile")
ax.set_xlabel("x")
ax.set_ylabel(r"$\rho$")
ax.legend(ncol = 1, fontsize = 'small')

ax = axs[1, 0]
im = ax.imshow(C1[:, : , nz//2])
ax.set_xlabel("y")
ax.set_ylabel("x")
plt.colorbar(im, ax = ax, label = r"$C_1$")

ax = axs[1, 1]
x = np.arange(0, nx, 1)
im = ax.plot(x, C1[:, ny//2 , nz//2], 'rx', label = "profile")
# ax.axhline(level, xmin = 0, xmax = 1, color = "k", ls = "-")
ax.set_xlabel("x")
ax.set_ylabel(r"$C_1$")
ax.legend(ncol = 1, fontsize = 'small')


fig.suptitle(f"$\kappa$ = {kappa}, $\chi = {chi}, T = {T}, t = {t}$")
fig.tight_layout()
plt.close()
fig

In [ ]:
# ts = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")
# times = np.zeros(len(ts))
# non_fluctuating_radii = np.zeros(len(ts))

# for i, ds in enumerate(ts):
#     times[i] = int(ds.current_time)
    
#     profile = read_amrex_data(ds, boxDim)
    
#     if variable == "phi":
#         to_plot = profile["phi"]
#     elif variable == "C1":
#         to_plot = (profile["phi"] + profile["rho"])/2

#     # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
#     curr_fit, _, _ = droplet_radius_profile_1d(to_plot, binned = False)
#     non_fluctuating_radii[i] = curr_fit[0]

#     # non_fluctuating_radii[i] = droplet_radius_mass(to_plot)

# ar = 1.25
# fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

# ax.plot(times, non_fluctuating_radii, color = colors[0], marker = markers[0], ls = "None", markerfacecolor = "None")
# ax.set_xlabel(r"$t \Delta t$")
# ax.set_ylabel(r"$R \Delta x$")
# ax.set_title(r"Droplet Radius before fluctuations switch on")
# plt.close()
# fig

In [ ]:
# np.gradient(non_fluctuating_radii)

In [ ]:
ts = yt.load(f"{savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for i, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[i] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # Axial fluctuations of gyration tensor
    # nbins = int(boxDim.min()*np.sqrt(3))
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, com, fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # level = curr_fit[2]

    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # R[i] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[i] = com
    dR[i] = axial_radii(filtered, com)
    # gr = gyration_tensor(com, filtered) # Calculating the gyration tensor of the droplet
    # egr = np.sqrt(np.linalg.eigvals(gr)) # calculating the unordered eigenvalues of the gyration tensor
    
    # curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', binned = False)
    R[i] = curr_fit[0]
    # R[i] = droplet_radius_mass(filtered)

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (ar*figheight*2, figheight))
alpha = 0.7
nbin = 15
edgecolor = "None"

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, edgecolor = edgecolor, bins = nbin, color = colors[0])
ax.axvline(np.mean(R), color = colors[0], ls = "--", lw = 1)

ax.set_title("Radius fluctuation")

dR2 = dR.copy()
# dR2 = np.power(dR2, 2)
dR2 -= 1.0
dR2 *= R.mean()

ax = axs[1]
for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                  label = f"$\delta${chr(i+97)}",
                                  alpha = alpha, edgecolor = edgecolor, 
                                  bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
plt.close()
print(sigma_fluct)
fig
# fig.savefig(f"{figures}/droplet-shape_flucuations-{noise_type}.png", dpi = 300)

Another test that can be performed to assess the accuracy of our model is to calculate the surface tension between fluids from the fluctuations in the shape of a spherical droplet. Similar to the flat interface case, fluctuations will induce roughening of the surface which will manifest as shape fluctuations of the droplet. This phenomena can be captured and the surface tension deduced. The calculated surface tension should match that calculated from the theoretical surface tension. To test this, we define the model parameters in this section to be $T=0.2, \chi = 0.42, \kappa = 0.01$, $k_bT = 10^{-7}$ and a system size of $64^3$. The surface tension calculated from theory at the used model parameters is $\sigma = 0.00135$. The droplet is initiated to have a radius $19.2 \Delta x$ and equilibrated until the radius change characterized is within machine precision. The fluctuations are switched on and run for $10^{5}$ timesteps. The final $5000$ timesteps are sampled to analyze the shape of the droplet. The density of the droplet phase $C_1(\mathbf{x}) = \frac{\rho(\mathbf{x}) + \phi(\mathbf{x})}{2}$ is used with the following filter

\begin{equation}
    C_1^{\text{filt}} =
    \begin{cases} 
      C_1 & \text{if } C_1 > 0.5 \\
      0 & \text{if } C_1 \leq 0.5
    \end{cases}
\end{equation}

The droplet radius, center of mass and gyration tensor are calculated from $C_1^{\text{filt}}$. We plot the droplet radius and axis fluctuations in Figure \ref{fig:droplet_radial_fluctuations_fig}.

\begin{figure}
    \centering
    \includegraphics[scale=0.5]{figures/droplet-shape_flucuations-spatially_independent.png}
    \caption{Plots of the droplet radius distribution (left) and the fluctuations in the radii of the principal axii (right).}
    \label{fig:droplet_radial_fluctuations_fig}
\end{figure}

Figure \ref{fig:droplet_radial_fluctuations_fig} shows that the mean fluctuating radius $\langle R \rangle = 17.983$ and the axial radius fluctuations are centered around 0. The calculated surface tension is $\sigma = 0.00133$ which represents a $ \approx 1.5 \%$ difference to the theoretical value from the thermodynamic parameters, indicating that the fluctuations are implemented correctly.

In [ ]:
ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (figheight*ar, figheight))

for i in range(3):
    ax.plot(times - times[0], dR2[:, i], color = colors[i], marker = markers[i], linestyle = "None",
            label = r"$\delta$" + chr(i+120), markerfacecolor = "None")

i += 1

ax.plot(times - times[0], np.sum(dR2, axis = -1), color = colors[i], marker = "None", linestyle = linestyles[i],
            label = "$\Sigma \delta R$", markerfacecolor = "None")

ax.legend(ncol = 2)
ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$\delta R \Delta x$")
plt.close()
fig.savefig(f"{figures}/droplet-shape_flucuations_time-{noise_type}.png", dpi = 300)
fig

In [ ]:
dR2 = dR.copy()
# dR2 = np.sqrt(dR2)
dR2 -= 1.0
dR2 *= R.mean()

# mode_20 = np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 0] + dR2[:, 1]), 2).mean() + np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 1] + dR2[:, 2]), 2).mean() + np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 0] + dR2[:, 2]), 2).mean()
# mode_22 = np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 0] - dR2[:, 1]), 2).mean() + np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 1] - dR2[:, 2]), 2).mean() + np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 0] - dR2[:, 2]), 2).mean()

mode_20 = np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 0] + dR2[:, 1]), 2) + np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 1] + dR2[:, 2]), 2) + np.power(-np.power(4*np.pi/5, 0.5)*(dR2[:, 0] + dR2[:, 2]), 2)
mode_22 = np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 0] - dR2[:, 1]), 2) + np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 1] - dR2[:, 2]), 2) + np.power(np.power(2*np.pi/15, 0.5)*(dR2[:, 0] - dR2[:, 2]), 2)

ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))
idx = 0

ax.plot(times - times[0], mode_20/3, label = r"$\zeta_{20}$", color = colors[idx], 
        marker = markers[idx], markerfacecolor = "None", linestyle = "None")
ax.axhline(mode_20.mean()/3, color = colors[idx])
idx += 1

ax.plot(times - times[0], mode_22*2/3, label = r"$\zeta_{2 \pm 2}$", color = colors[idx], 
        marker = markers[idx], markerfacecolor = "None", linestyle = "None")
ax.axhline(mode_22.mean()*2/3, color = colors[idx])
idx += 1

ax.axhline(kbt/(4*sigma_theory), color = colors[idx], label = "theory")

mode_20.mean()/3/2, (mode_22.mean() + mode_22.mean())/3/2, kbt/(4*sigma_theory)

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$| \zeta ^2 | \Delta x^2$")
ax.legend()

ax.set_title("Mode fluctuations")

plt.close()
fig.savefig(f"{figures}/spherical-mode_flucuations_time-{noise_type}.png", dpi = 300)
fig

In [ ]:
ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

lns = []

ln, = ax.plot(times - times[0], R, color = colors[0], ls = "None", 
            marker = markers[0], markerfacecolor = "None", ms = 8, label = "R")
lns.append(ln)

ln = ax.axhline(y = R.mean(), xmin = 0, xmax = 1, color = "magenta", label = r"$\langle R \rangle$")
lns.append(ln)


ts = yt.load(f"{savedir}/non_fluctuating/hydro_plt*")
profile = read_amrex_data(ts[-1], boxDim)

if variable == "phi":
    to_plot = profile['phi']
elif variable == "C1":
    to_plot = (profile['phi'] + profile['rho'])/2

to_plot = np.where(to_plot > level, to_plot, 0)
curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
non_fluctuating_radius = curr_fit[0]

ln = ax.axhline(y = non_fluctuating_radius, xmin = 0, xmax = 1, color = "lime", label = r"$R_{det}$")
lns.append(ln)

print(non_fluctuating_radius - R.mean())

# print(f"{(R_all.mean() - non_fluctuating_radius)/non_fluctuating_radius*100:.3f}% difference between fluctuating and non fluctuating radius")

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R_{d} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Droplet radius")

plt.close()
fig
# fig.savefig(f"{figures}/droplet-radius_fluctuating_time-{noise_type}.png", dpi = 300)
fig

### Varying kappa (in progress)

In [ ]:
# Details box dimensions and information location (long run and big system)
nx = 64
ny = 64
nz = 64
boxDim = np.array([nx, ny, nz])
nbins = boxDim.min()

# thermodynamic information used to conduct simulations
chi = 0.42
T = 0.2
kappas = [0.005, 0.01, 0.02, 0.03]
kbt = 1e-7
noise_type = "spatially_independent"
variable = "C1"

# sigma_theory = swift_theoretical_sigma(chi/T, kappa)
all_sigma_theory = [swift_critical_sigma(chi, T, kappa) for kappa in kappas]
# sigma_theory = swift_critical_sigma(chi, T, kappa)
C0 = swift_theoretical_C0(chi/T)
# zeta = swift_theoretical_xi(C0, chi/T, kappa)

idx = -1

savedir = './validation/droplet_fluctuations/spatially_independent/'
filename = "fluctuating_on"

output_file = "hydro_plt"
center_slc = np.s_[nx//2-1:nx//2+2, ny//2-1:ny//2+2, nz//2-1:nz//2+2]
edge_slc = np.s_[0:nx:nx-1, 0:ny:ny-1, 0:nz:nz-1]

In [ ]:
all_times = []
all_non_fluctuating_radii = []

for i, kappa in enumerate(kappas):
    curr_savedir = f"{savedir}/kappa_{kappa}"

    ts = yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")
    times = np.zeros(len(ts))
    non_fluctuating_radii = np.zeros(len(ts))

    for j, ds in enumerate(ts):
        times[j] = int(ds.current_time)
        
        profile = read_amrex_data(ds, boxDim)
        
        if variable == "phi":
            to_plot = profile["phi"]
        elif variable == "C1":
            to_plot = (profile["phi"] + profile["rho"])/2

        curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
        non_fluctuating_radii[j] = curr_fit[0]

        # non_fluctuating_radii[i] = droplet_radius_mass(to_plot)
    
    all_times.append(times)
    all_non_fluctuating_radii.append(non_fluctuating_radii)


ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))

for i in range(len(kappas)):
    kappa = kappas[i]
    times = all_times[i]
    non_fluctuating_radii = all_non_fluctuating_radii[i]

    ax.plot(times, non_fluctuating_radii, color = colors[i], label = f"$\kappa = {kappa}$",
            marker = markers[i], ls = "None", markerfacecolor = "None")
    
    grad_radii = np.gradient(non_fluctuating_radii)
    idxs = np.where(np.abs(grad_radii) < 1e-9)[0]

    if len(idxs) > 0:
        ax.axvline(times[idxs[0]], color = colors[i], linestyle = linestyles[i])

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R \Delta x$")
ax.set_title(r"Droplet Radius before fluctuations switch on")
ax.legend()
plt.close()

fig

In [ ]:
np.gradient(all_non_fluctuating_radii[0])

In [ ]:
np.gradient(all_non_fluctuating_radii[1])

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (2*figheight*ar, figheight))
alpha = 0.7
nbin = 15
edgecolor = "None"
kappa_idx = 0

kappa = kappas[kappa_idx]
sigma_theory = all_sigma_theory[kappa_idx]

lc = np.sqrt(kappa/(2*(chi/T)))
l_star = np.sqrt(kbt/sigma_theory)

curr_savedir = f"{savedir}/kappa_{kappa}"
ts = yt.load(f"{curr_savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for j, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[j] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # Axial fluctuations of gyration tensor
    # nbins = int(boxDim.min()*np.sqrt(3))
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, com, fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # level = curr_fit[2]

    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # R[i] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[j] = com
    dR[j] = axial_radii(filtered, com)

    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    R[j] = curr_fit[0]

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, edgecolor = edgecolor, bins = nbin, color = colors[0])
ax.axvline(np.mean(R), color = colors[0], ls = "--", lw = 1)

ax.set_title("Radius fluctuation")

dR2 = dR.copy()
# dR2 = np.power(dR2, 2)
dR2 -= 1.0
dR2 *= R.mean()

ax = axs[1]
for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                label = f"$\delta${chr(i+120)}",
                                alpha = alpha, edgecolor = edgecolor, 
                                bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
# sigmas_real.append(sigma_fluct)
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
print(r"$l^{*}$="+ f"{l_star:.3e}")
print(r"$l_{c}$="+ f"{lc:.3e}")

plt.close()

fig

In [ ]:
ar = 1.25
fig, axs = plt.subplots(1, 2, figsize = (2*ar*figheight, figheight))


ax = axs[0]
lns = []

# ln, = ax.plot(times - times[0], R, color = colors[0], ls = "None", 
#             marker = markers[0], markerfacecolor = "None", ms = 8, label = "R")
# lns.append(ln)

# ln = ax.axhline(y = R.mean(), xmin = 0, xmax = 1, color = "magenta", label = r"$\langle R \rangle$")
# lns.append(ln)

profile = read_amrex_data(yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")[-1], boxDim)

if variable == "phi":
    to_plot_non_fluctuating = profile['phi']
elif variable == "C1":
    to_plot_non_fluctuating = (profile['phi'] + profile['rho'])/2

curr_fit, _, _ = droplet_radius_profile_1d(np.where(to_plot_non_fluctuating > level, to_plot, 0), fit = 'tanh', bins = nbins)
non_fluctuating_radius = curr_fit[0]

ln = ax.axhline(y = non_fluctuating_radius, xmin = 0, xmax = 1, color = "lime", label = r"$R_{det}$")
lns.append(ln)

# print(f"{(R_all.mean() - non_fluctuating_radius)/non_fluctuating_radius*100:.3f}% difference between fluctuating and non fluctuating radius")

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R_{d} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Droplet radius")


ax = axs[1]

ax.plot(to_plot_non_fluctuating[nx//2, ny//2], 'rx', label = "non fluctuating")
ax.plot(to_plot[nx//2, ny//2], 'bo', label = "fluctuating", markerfacecolor = "None")

plt.close()
fig
# fig.savefig(f"{figures}/droplet-radius_fluctuating_time-{noise_type}.png", dpi = 300)

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (2*figheight*ar, figheight))
alpha = 0.7
nbin = 15
edgecolor = "None"
kappa_idx = 1
variable = "C1"

sigma_theory = all_sigma_theory[kappa_idx]
kappa = kappas[kappa_idx]
curr_savedir = f"{savedir}/kappa_{kappa}"

lc = np.sqrt(kappa/(2*(chi/T)))
l_star = np.sqrt(kbt/sigma_theory)

ts = yt.load(f"{curr_savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for j, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[j] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # Axial fluctuations of gyration tensor
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[j] = curr_fit[0]
    # R[j] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    # level = ski.filters.threshold_otsu(to_plot)
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[j] = com
    dR[j] = axial_radii(filtered, com)

    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    R[j] = curr_fit[0]

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, edgecolor = edgecolor, bins = nbin, color = colors[0])
ax.axvline(np.mean(R), color = colors[0], ls = "--", lw = 1)

ax.set_title("Radius fluctuation")

dR2 = dR.copy()
dR2 -= 1.0
dR2 *= R.mean()

ax = axs[1]
for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                label = f"$\delta${chr(i+120)}",
                                alpha = alpha, edgecolor = edgecolor, 
                                bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
# sigmas_real.append(sigma_fluct)
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
print(r"$l^{*}$="+ f"{l_star:.3e}")
print(r"$l_{c}$="+ f"{lc:.3e}")

plt.close()
fig

In [ ]:
ar = 1.25
fig, axs = plt.subplots(1, 2, figsize = (2*ar*figheight, figheight))


ax = axs[0]
lns = []

ln, = ax.plot(times - times[0], R, color = colors[0], ls = "None", 
            marker = markers[0], markerfacecolor = "None", ms = 8, label = "R")
lns.append(ln)

ln = ax.axhline(y = R.mean(), xmin = 0, xmax = 1, color = "magenta", label = r"$\langle R \rangle$")
lns.append(ln)

curr_savedir = f"{savedir}/kappa_{kappa}"
profile = read_amrex_data(yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")[-1], boxDim)

if variable == "phi":
    to_plot_non_fluctuating = profile['phi']
elif variable == "C1":
    to_plot_non_fluctuating = (profile['phi'] + profile['rho'])/2


to_plot_non_fluctuating = np.where(to_plot_non_fluctuating > level, to_plot, 0)
curr_fit, _, _ = droplet_radius_profile_1d(to_plot_non_fluctuating, fit = 'tanh', bins = nbins)
non_fluctuating_radius = curr_fit[0]
# non_fluctuating_radius = droplet_radius_mass(to_plot_non_fluctuating)

ln = ax.axhline(y = non_fluctuating_radius, xmin = 0, xmax = 1, color = "lime", label = r"$R_{det}$")
lns.append(ln)

# print(f"{(R_all.mean() - non_fluctuating_radius)/non_fluctuating_radius*100:.3f}% difference between fluctuating and non fluctuating radius")

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R_{d} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Droplet radius")


ax = axs[1]

ax.plot(to_plot_non_fluctuating[nx//2, ny//2], 'rx', label = "non fluctuating")
ax.plot(filtered[nx//2, ny//2], 'bo', label = "fluctuating", markerfacecolor = "None")

ax.legend()
ax.set_xlabel("z")
ax.set_ylabel(variable)

plt.close()
fig
# fig.savefig(f"{figures}/droplet-radius_fluctuating_time-{noise_type}.png", dpi = 300)

In [ ]:
# 0.003582890757007382 0.0019047619047618996 0.8810176474288808 C1
# 0.003511284381190737 0.0019047619047618996 0.8434243001251419

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (2*figheight*ar, figheight))
alpha = 0.7
nbin = 15
edgecolor = "None"
kappa_idx = 2
variable = "C1"

kappa = kappas[kappa_idx]
sigma_theory = all_sigma_theory[kappa_idx]
lc = np.sqrt(kappa/(2*(chi/T)))
l_star = np.sqrt(kbt/sigma_theory)


curr_savedir = f"{savedir}/kappa_{kappa}/old"
ts = yt.load(f"{curr_savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for j, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[j] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # Axial fluctuations of gyration tensor
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # level = curr_fit[2]
    # R[j] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    # level = ski.filters.threshold_otsu(to_plot)
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[j] = com
    dR[j] = axial_radii(filtered, com)

    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    R[j] = curr_fit[0]

ax = axs[0]
bins, counts, patch = ax.hist(R, density = True, alpha = alpha, edgecolor = edgecolor, bins = nbin, color = colors[0])
ax.axvline(np.mean(R), color = colors[0], ls = "--", lw = 1)

ax.set_title("Radius fluctuation")

dR2 = dR.copy()
dR2 -= 1.0
dR2 *= R.mean()

ax = axs[1]
for i in range(3):
    bins, counts, patch = ax.hist(dR2[:, i], density = True, 
                                label = f"$\delta${chr(i+120)}",
                                alpha = alpha, edgecolor = edgecolor, 
                                bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
# sigmas_real.append(sigma_fluct)
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
print(r"$l^{*}$="+ f"{l_star:.3e}")
print(r"$l_{c}$="+ f"{lc:.3e}")

plt.close()
fig

In [ ]:
ar = 1.25
fig, axs = plt.subplots(1, 2, figsize = (2*ar*figheight, figheight))


ax = axs[0]
lns = []

ln, = ax.plot(times - times[0], R, color = colors[0], ls = "None", 
            marker = markers[0], markerfacecolor = "None", ms = 8, label = "R")
lns.append(ln)

ln = ax.axhline(y = R.mean(), xmin = 0, xmax = 1, color = "magenta", label = r"$\langle R \rangle$")
lns.append(ln)

curr_savedir = f"{savedir}/kappa_{kappa}"
profile = read_amrex_data(yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")[-1], boxDim)

if variable == "phi":
    to_plot_non_fluctuating = profile['phi']
elif variable == "C1":
    to_plot_non_fluctuating = (profile['phi'] + profile['rho'])/2


to_plot_non_fluctuating = np.where(to_plot_non_fluctuating > level, to_plot, 0)
curr_fit, _, _ = droplet_radius_profile_1d(to_plot_non_fluctuating, fit = 'tanh', bins = nbins)
non_fluctuating_radius = curr_fit[0]
# non_fluctuating_radius = droplet_radius_mass(to_plot_non_fluctuating)

ln = ax.axhline(y = non_fluctuating_radius, xmin = 0, xmax = 1, color = "lime", label = r"$R_{det}$")
lns.append(ln)

# print(f"{(R_all.mean() - non_fluctuating_radius)/non_fluctuating_radius*100:.3f}% difference between fluctuating and non fluctuating radius")

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R_{d} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Droplet radius")


ax = axs[1]

ax.plot(to_plot_non_fluctuating[nx//2, ny//2], 'rx', label = "non fluctuating")
ax.plot(filtered[nx//2, ny//2], 'bo', label = "fluctuating", markerfacecolor = "None")

ax.legend()
ax.set_xlabel("z")
ax.set_ylabel(variable)

plt.close()
fig
# fig.savefig(f"{figures}/droplet-radius_fluctuating_time-{noise_type}.png", dpi = 300)

In [ ]:
kappa_idx = 3
variable = "C1"

kappa = kappas[kappa_idx]
sigma_theory = all_sigma_theory[kappa_idx]
lc = np.sqrt(kappa/(2*(chi/T)))
l_star = np.sqrt(kbt/sigma_theory)


curr_savedir = f"{savedir}/kappa_{kappa}/"
ts = yt.load(f"{curr_savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for j, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[j] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # Axial fluctuations of gyration tensor
    # nbins = int(boxDim.min()*np.sqrt(3))
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, com, fit = 'tanh', bins = nbins)
    # R[j] = curr_fit[0]
    # level = curr_fit[2]

    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # R[j] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    # level = ski.filters.threshold_otsu(to_plot)
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[j] = com
    dR[j] = axial_radii(filtered, com)

    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    R[j] = curr_fit[0]

In [ ]:
ar = 1
fig, axs = plt.subplots(1, 2, figsize = (2*figheight*ar, figheight))
alpha = 0.7
nbin = 15
edgecolor = "None"

ax = axs[0]
bins, counts, patch = ax.hist(R[2:], density = True, alpha = alpha, edgecolor = edgecolor, bins = nbin, color = colors[0])
ax.axvline(np.mean(R), color = colors[0], ls = "--", lw = 1)

ax.set_title("Radius fluctuation")

dR2 = dR.copy()
dR2 -= 1.0
dR2 *= R.mean()

ax = axs[1]
for i in range(3):
    bins, counts, patch = ax.hist(dR2[2:, i], density = True, 
                                label = f"$\delta${chr(i+120)}",
                                alpha = alpha, edgecolor = edgecolor, 
                                bins = nbin, color = colors[i])

ax.set_title("Axial fluctuation")

ax.legend()

sigma_fluct = droplet_fluctuations(dR2, temp = kbt) # y20 y22
# sigmas_real.append(sigma_fluct)
print(np.mean(sigma_fluct), sigma_theory, np.mean(np.mean(sigma_fluct) - sigma_theory)/(sigma_theory))
print(r"$l^{*}$="+ f"{l_star:.3e}")
print(r"$l_{c}$="+ f"{lc:.3e}")

plt.close()
fig

In [ ]:
ar = 1.25
fig, axs = plt.subplots(1, 2, figsize = (2*ar*figheight, figheight))


ax = axs[0]
lns = []

ln, = ax.plot(times - times[0], R, color = colors[0], ls = "None", 
            marker = markers[0], markerfacecolor = "None", ms = 8, label = "R")
lns.append(ln)

ln = ax.axhline(y = R.mean(), xmin = 0, xmax = 1, color = "magenta", label = r"$\langle R \rangle$")
lns.append(ln)

curr_savedir = f"{savedir}/kappa_{kappa}"
profile = read_amrex_data(yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")[-1], boxDim)

if variable == "phi":
    to_plot_non_fluctuating = profile['phi']
elif variable == "C1":
    to_plot_non_fluctuating = (profile['phi'] + profile['rho'])/2


to_plot_non_fluctuating = np.where(to_plot_non_fluctuating > level, to_plot, 0)
curr_fit, _, _ = droplet_radius_profile_1d(to_plot_non_fluctuating, fit = 'tanh', bins = nbins)
non_fluctuating_radius = curr_fit[0]
# non_fluctuating_radius = droplet_radius_mass(to_plot_non_fluctuating)

ln = ax.axhline(y = non_fluctuating_radius, xmin = 0, xmax = 1, color = "lime", label = r"$R_{det}$")
lns.append(ln)

# print(f"{(R_all.mean() - non_fluctuating_radius)/non_fluctuating_radius*100:.3f}% difference between fluctuating and non fluctuating radius")

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$R_{d} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Droplet radius")


ax = axs[1]

ax.plot(to_plot_non_fluctuating[nx//2, ny//2], 'rx', label = "non fluctuating")
ax.plot(filtered[nx//2, ny//2], 'bo', label = "fluctuating", markerfacecolor = "None")

ax.legend()
ax.set_xlabel("z")
ax.set_ylabel(variable)

plt.close()
fig
# fig.savefig(f"{figures}/droplet-radius_fluctuating_time-{noise_type}.png", dpi = 300)

In [ ]:
kappa = kappas[2]
curr_savedir = f"{savedir}/kappa_{kappa}"
ts = yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")
ds = ts[-1]
profile = read_amrex_data(ds, boxDim)

rho = profile["rho"]
phi = profile["phi"]
C1 = (rho+phi)/2

radii = droplet_radius_mass(C1)

# derived from moments
Pxx = 1/3*(profile["mf4"] + profile["mf5"]) + rho*1/3
Pyy = 1/3*(profile["mf4"] - 1/2*profile["mf5"] + profile["mf6"]) + rho*1/3
Pzz = 1/3*(profile["mf4"] - 1/2*profile["mf5"] - profile["mf6"]) + rho*1/3
scp = (Pxx + Pyy + Pzz)/3

pressure = pressure_jump(scp)
pressure*radii/2

In [ ]:
# curr_savedir = f"{savedir}/kappa_0.03"
# ts = yt.load(f"{curr_savedir}/non_fluctuating/hydro_plt*")
# ds = ts[0]

# profile = read_amrex_data(ds, boxDim)
# phi = profile["phi"]
# C1 = (profile["phi"] + profile["rho"])/2

# to_plot = C1

# plt.imshow(to_plot[nx//2])
# plt.show()

# params, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), bins = nx)
# params[0]

### MSD

In [ ]:
ar = 1.25
fig, ax = plt.subplots(1, 1, figsize = (ar*figheight, figheight))
lns = []

idx = 0
ln, = ax.semilogx(times - times[0], coms_all[:, idx - 1], color = colors[idx], ls = linestyles[idx], 
            marker = "None", markerfacecolor = "None", ms = 8, label = chr(idx+120))
lns.append(ln)
idx += 1
ln, = ax.semilogx(times - times[0], coms_all[:, idx - 1], color = colors[idx], ls = linestyles[idx], 
            marker = "None", markerfacecolor = "None", ms = 8, label = chr(idx+120))
lns.append(ln)
idx += 1
ln, = ax.semilogx(times - times[0], coms_all[:, idx - 1], color = colors[idx], ls = linestyles[idx], 
            marker = "None", markerfacecolor = "None", ms = 8, label = chr(idx+120))
lns.append(ln)

ax.set_xlabel(r"$t \Delta t$")
ax.set_ylabel(r"$x^{cm}_{i} \Delta x$")
ax.legend(handles = lns, ncol = 3)
ax.set_title("Center of mass location")
plt.close()

In [ ]:
fig

In [ ]:
# Details box dimensions and information location (long run and big system)
nx = 64
ny = 64
nz = 64
boxDim = np.array([nx, ny, nz])
nbins = boxDim.min()

# thermodynamic information used to conduct simulations
chi = 0.42
T = 0.2
kappa = 0.01
kbt = 1e-7
noise_type = "spatially_independent"
variable = "C1"

sigma_theory = swift_critical_sigma(chi, T, kappa)
C0 = swift_theoretical_C0(chi/T)
zeta = swift_theoretical_xi(C0, chi/T, kappa)

idx = -1

savedir = f"./validation/droplet_fluctuations/spatially_independent/kappa_{kappa}/"
filename = "fluctuating_on"

output_file = "hydro_plt"
center_slc = np.s_[nx//2-1:nx//2+2, ny//2-1:ny//2+2, nz//2-1:nz//2+2]
edge_slc = np.s_[0:nx:nx-1, 0:ny:ny-1, 0:nz:nz-1]

In [ ]:
ts = yt.load(f"{savedir}/hydro_plt*")[1:]

times = np.zeros(len(ts))
R = np.zeros(len(ts))
# dR = np.zeros((len(ts), 3))
coms_all = np.zeros((len(ts), 3))

for i, ds in enumerate(ts):
    profile = read_amrex_data(ds, boxDim)
    rho = profile["rho"]
    phi = profile["phi"]
    C1 = (rho+phi)/2

    times[i] = int(ds.current_time)

    if variable == "phi":
        to_plot = phi
        level = 0
    elif variable == "C1":
        to_plot = C1
        level = 0.5

    # to_plot = phi
    # level = 0
    # Axial fluctuations of gyration tensor
    # nbins = int(boxDim.min()*np.sqrt(3))
    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, com, fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # level = curr_fit[2]

    # curr_fit, _, _ = droplet_radius_profile_1d(to_plot, center_of_mass(to_plot), fit = 'tanh', bins = nbins)
    # R[i] = curr_fit[0]
    # R[i] = droplet_radius_mass(to_plot)

    # filtered = to_plot
    filtered = np.where(to_plot > level, to_plot, 0)
    # filtered = sharpen_droplet_interface(to_plot, int_height = level, max_val = 2*C0-1, min_val = 0)

    com = center_of_mass(filtered)
    coms_all[i] = com
    # dR[i] = axial_radii(filtered, com)

    curr_fit, _, _ = droplet_radius_profile_1d(filtered, com, fit = 'tanh', bins = nbins)
    R[i] = curr_fit[0]

In [ ]:
fig, ax = plt.subplots(1, 1, figsize = (5, 4))

time_lag_sz = 13

alpha = 1 # viscosity ratio
fn = (6 + 4*alpha)/(1 + alpha)
L = np.mean(boxDim)
eta_0 = 1/3*(0.7886751345948129 - 0.5)*1
periodicity = 1 - 2.84*(R.mean()/L)
correction_factor = periodicity/(fn*np.pi*eta_0*R.mean())
Dse = correction_factor*kbt

pos_vars = np.zeros((3, time_lag_sz))
D_s = np.zeros(3)

for i in range(3):
    lag_times, pos_var = msd_with_time_lag_1d(coms_all[:, i], window_sz = time_lag_sz, time_int = 50)
    # lag_times, pos_var = calculate_msd_with_lag(coms_all[:, i], time=times - times[0], max_lag=time_lag_sz)
    pos_vars[i] = pos_var
    ax.plot(lag_times, pos_var, color = colors[i], ls = "None", marker = markers[i], label = chr(120+i), markerfacecolor = "None")

    popt = np.polyfit(lag_times[1:], pos_var[1:], deg = 1)
    ax.plot(lag_times, popt[0]*lag_times + popt[1], linestyle = linestyles[i])

    # popt, _ = curve_fit(lambda x, a: a*x, lag_times, pos_var, p0 = (Dse))
    # ax.plot(lag_times, popt[0]*lag_times, linestyle = linestyles[i])

    D_s[i] = popt[0]/2

i += 1
ax.plot(lag_times, lag_times*Dse, color = colors[i], ls = None, marker = markers[i], label = "Stokes-Einstein", markerfacecolor = "None")

ax.set_xlabel(r"$\tau$")
ax.set_ylabel(r"$\langle (\vec{r}(t + \tau) - \vec{r})^2 \rangle$")

print(f"Dse: {Dse:.3e}, Db: {np.mean(D_s):.3e}, diff: {(np.mean(D_s) - Dse)/Dse*100:.2f}%")

ax.legend()
ax.set_title("MSD of droplet center of mass")
# fig.savefig(f"{figures}/MSD_droplet-{noise_type}.png", dpi = 300)